# Projective Scheme Framework v2

We construct slice and coslice categories as Sage functorial constructions, route universal constructions through certified backends, and verify an affine chart of the universal K3 double-cover family after base change to the exact fixed-point-free open in the full invariant parameter space.

In [1]:
FRAMEWORK_V2_NAME = 'Projective Scheme Framework v2'
FRAMEWORK_V2_SCOPE = (
    'category-first slice/coslice infrastructure with routed computational backends'
)
print(FRAMEWORK_V2_NAME)
print(FRAMEWORK_V2_SCOPE)

Projective Scheme Framework v2
category-first slice/coslice infrastructure with routed computational backends


In [2]:
from sage.all import *
from sage.categories.category import Category
from sage.categories.sets_cat import Sets
from sage.categories.covariant_functorial_construction import (
    CovariantFunctorialConstruction,
    CovariantConstructionCategory,
)
from sage.categories.homset import Homset, Hom
from sage.categories.morphism import Morphism
from sage.categories.map import Map, FormalCompositeMap
from sage.categories.commutative_rings import CommutativeRings
from sage.categories.schemes import Schemes
from sage.schemes.generic.spec import Spec as _sage_Spec
from sage.misc.cachefunc import cached_method
from sage.rings.polynomial.multi_polynomial_ring_base import (
    MPolynomialRing_base,
)
from sage.rings.polynomial.polynomial_ring import (
    PolynomialRing_generic,
)
from sage.structure.element import Element
from sage.structure.parent import Parent
from sage.structure.sage_object import SageObject

print('Sage version =', sage.version.version)

Sage version = 10.10.beta0


## Arrow, slice, and coslice categories

For a category $\mathcal C$, the arrow category $\operatorname{Ar}(\mathcal C)=\mathcal C^{[1]}$ has morphisms of $\mathcal C$ as objects and commuting squares as morphisms. The coslice $(R\downarrow\mathcal C)$ and slice $(\mathcal C\downarrow S)$ are strict fibers of the source and target functors: their morphisms are commuting squares whose fixed leg is the appropriate identity.

In [3]:
def _native_equality_proves_equal(left, right):
    if left is right:
        return True
    try:
        return True if left == right else None
    except (TypeError, ValueError, AttributeError, NotImplementedError):
        return None


def _morphism_is_identity(morphism):
    if hasattr(morphism, 'is_identity'):
        try:
            return True if morphism.is_identity() else False
        except (TypeError, ValueError, AttributeError, NotImplementedError):
            pass
    return None


class MorphismEqualityCertificate(SageObject):
    def domain(self):
        raise NotImplementedError

    def verify(self, left, right):
        raise NotImplementedError


class PolynomialGeneratorEqualityCertificate(MorphismEqualityCertificate):
    def __init__(self, polynomial_ring):
        if not isinstance(
            polynomial_ring,
            (MPolynomialRing_base, PolynomialRing_generic),
        ):
            raise TypeError(
                'the generator certificate requires a polynomial-ring domain'
            )
        self._domain = polynomial_ring

    def domain(self):
        return self._domain

    def verify(self, left, right):
        if left.domain() is not self._domain:
            return False
        if right.domain() is not self._domain:
            return False
        if left.codomain() is not right.codomain():
            return False
        if not all(
            left(generator) == right(generator)
            for generator in self._domain.gens()
        ):
            return False

        coefficient_ring = self._domain.base_ring()
        if coefficient_ring in (QQ, ZZ):
            return True
        coefficient_inclusion = self._domain.coerce_map_from(
            coefficient_ring
        )
        if coefficient_inclusion is None:
            return None
        left_on_coefficients = left * coefficient_inclusion
        right_on_coefficients = right * coefficient_inclusion
        if left_on_coefficients == right_on_coefficients:
            return True
        if isinstance(
            coefficient_ring,
            (MPolynomialRing_base, PolynomialRing_generic),
        ):
            return PolynomialGeneratorEqualityCertificate(
                coefficient_ring
            ).verify(
                left_on_coefficients,
                right_on_coefficients,
            )
        return None


class SquareCommutativityCertificate(SageObject):
    def applies(self, source_arrow, target_arrow, source_leg, target_leg):
        raise NotImplementedError

    def verify(self, source_arrow, target_arrow, source_leg, target_leg):
        raise NotImplementedError


class ArrowCategoryFunctor(CovariantFunctorialConstruction):
    _functor_name = 'arrow_object'
    _functor_category = 'ArrowCategory'


class ArrowCategoryConstruction(CovariantConstructionCategory):
    _functor_category = 'ArrowCategory'
    _base_category_class = (Category,)

    def __init__(self, category):
        self._base_category = category
        self._args = tuple()
        self._morphism_equality_certificates = {}
        self._square_certificates = []
        Category.__init__(self)

    def _repr_object_names(self):
        return (
            'arrows in '
            f'{self.base_category()._repr_object_names()}'
        )

    def arrow_category(self):
        return self

    def arrow_object(self, arrow):
        return ArrowObject(
            self,
            arrow,
        )

    def arrows(self, source, target):
        return Hom(
            source,
            target,
            category=self.base_category(),
        )

    def register_morphism_equality_certificate(self, certificate):
        self._morphism_equality_certificates[
            id(certificate.domain())
        ] = certificate
        return self

    def register_square_certificate(self, certificate):
        self._square_certificates.append(certificate)
        return self

    def morphisms_equal(self, left, right):
        if _native_equality_proves_equal(left, right) is True:
            return True
        if left.domain() is not right.domain():
            return False
        if left.codomain() is not right.codomain():
            return False
        if (
            _morphism_is_identity(left) is True
            and _morphism_is_identity(right) is True
        ):
            return True
        certificate = self._morphism_equality_certificates.get(
            id(left.domain())
        )
        if certificate is None:
            return None
        return certificate.verify(left, right)

    def square_status(
        self,
        source_arrow,
        target_arrow,
        source_leg,
        target_leg,
    ):
        for certificate in self._square_certificates:
            if certificate.applies(
                source_arrow,
                target_arrow,
                source_leg,
                target_leg,
            ):
                status = certificate.verify(
                    source_arrow,
                    target_arrow,
                    source_leg,
                    target_leg,
                )
                if status is not None:
                    return status
        try:
            left_composite = target_leg * source_arrow
            right_composite = target_arrow * source_leg
        except (TypeError, ValueError, AttributeError, NotImplementedError):
            return None
        return self.morphisms_equal(
            left_composite,
            right_composite,
        )


@cached_method
def _arrow_category(self):
    return ArrowCategoryConstruction(self)


Category.ArrowCategory = _arrow_category


class CosliceCategoryFunctor(CovariantFunctorialConstruction):
    _functor_name = 'coslice_object'
    _functor_category = 'CosliceCategory'

    def __init__(self, base_object):
        self._base_object = base_object

    def base_object(self):
        return self._base_object


class CosliceCategoryConstruction(CovariantConstructionCategory):
    _functor_category = 'CosliceCategory'
    _base_category_class = (Category,)

    def __init__(self, category, base_object):
        if base_object not in category:
            raise TypeError(
                'the fixed object is not an object of the base category'
            )
        self._base_category = category
        self._args = (base_object,)
        self._base_object = base_object
        Category.__init__(self)

    def base_object(self):
        return self._base_object

    def arrow_category(self):
        return self.base_category().ArrowCategory()

    def extra_super_categories(self):
        return [self.arrow_category()]

    def _repr_object_names(self):
        return (
            f'objects under {self._base_object} in '
            f'{self.base_category()._repr_object_names()}'
        )

    def coslice_object(self, structure_morphism):
        if structure_morphism.domain() is not self._base_object:
            raise ValueError(
                'a coslice object must have the fixed source'
            )
        return CosliceObject(
            self,
            structure_morphism,
        )

    def initial_object(self):
        identity = self._base_object.Hom(
            self._base_object
        ).identity()
        return self.coslice_object(identity)

    def register_morphism_equality_certificate(self, certificate):
        self.arrow_category().register_morphism_equality_certificate(
            certificate
        )
        return self

    def register_square_certificate(self, certificate):
        self.arrow_category().register_square_certificate(
            certificate
        )
        return self


@cached_method
def _coslice_category(self, base_object):
    return CosliceCategoryConstruction(self, base_object)


Category.CosliceCategory = _coslice_category


class SliceCategoryFunctor(CovariantFunctorialConstruction):
    _functor_name = 'slice_object'
    _functor_category = 'SliceCategory'

    def __init__(self, base_object):
        self._base_object = base_object

    def base_object(self):
        return self._base_object


class SliceCategoryConstruction(CovariantConstructionCategory):
    _functor_category = 'SliceCategory'
    _base_category_class = (Category,)

    def __init__(self, category, base_object):
        if base_object not in category:
            raise TypeError(
                'the fixed object is not an object of the base category'
            )
        self._base_category = category
        self._args = (base_object,)
        self._base_object = base_object
        Category.__init__(self)

    def base_object(self):
        return self._base_object

    def arrow_category(self):
        return self.base_category().ArrowCategory()

    def extra_super_categories(self):
        return [self.arrow_category()]

    def _repr_object_names(self):
        return (
            f'objects over {self._base_object} in '
            f'{self.base_category()._repr_object_names()}'
        )

    def slice_object(self, structure_morphism):
        if structure_morphism.codomain() is not self._base_object:
            raise ValueError(
                'a slice object must have the fixed target'
            )
        return SliceObject(
            self,
            structure_morphism,
        )

    def terminal_object(self):
        identity = self._base_object.Hom(
            self._base_object
        ).identity()
        return self.slice_object(identity)

    def register_morphism_equality_certificate(self, certificate):
        self.arrow_category().register_morphism_equality_certificate(
            certificate
        )
        return self

    def register_square_certificate(self, certificate):
        self.arrow_category().register_square_certificate(
            certificate
        )
        return self


@cached_method
def _slice_category(self, base_object):
    return SliceCategoryConstruction(self, base_object)


Category.SliceCategory = _slice_category


class ArrowObject(SageObject):
    _arrow_kind = 'arrow'

    def __init__(
        self,
        category,
        arrow,
    ):
        if not hasattr(arrow, 'domain'):
            raise TypeError(
                'an arrow object requires a categorical morphism'
            )
        if not hasattr(arrow, 'codomain'):
            raise TypeError(
                'an arrow object requires a categorical morphism'
            )
        if arrow.domain() not in category.base_category():
            raise TypeError(
                'the arrow source is not in the base category'
            )
        if arrow.codomain() not in category.base_category():
            raise TypeError(
                'the arrow target is not in the base category'
            )
        self._arrow = arrow
        self._construction_category = category
        self._presentations = {}

    def category(self):
        return self._construction_category

    def arrow(self):
        return self._arrow

    def construction_category(self):
        return self._construction_category

    def arrow_kind(self):
        return self._arrow_kind

    def arrow_category(self):
        return self._construction_category.arrow_category()

    def source(self):
        return self._arrow.domain()

    def target(self):
        return self._arrow.codomain()

    def structure_morphism(self):
        return self._arrow

    def underlying_object(self):
        if self._arrow_kind == 'coslice':
            return self.target()
        if self._arrow_kind == 'slice':
            return self.source()
        raise AttributeError(
            'a general arrow has no distinguished underlying endpoint'
        )

    def register_presentation(self, presentation):
        self._presentations[presentation.kind] = presentation
        return self

    def presentation(self, kind):
        return self._presentations.get(kind)

    def _repr_(self):
        return f'Arrow object ({self._arrow})'

    def _Hom_(self, codomain, category=None):
        if self._arrow_kind == 'coslice':
            return CosliceHomset(self, codomain)
        if self._arrow_kind == 'slice':
            return SliceHomset(self, codomain)
        return ArrowHomset(self, codomain)


class CosliceObject(ArrowObject):
    _arrow_kind = 'coslice'


class SliceObject(ArrowObject):
    _arrow_kind = 'slice'


class ArrowHomset(Homset):
    def __init__(self, domain, codomain):
        if domain.construction_category() is not (
            codomain.construction_category()
        ):
            raise TypeError(
                'arrow morphisms require a common arrow construction category'
            )
        Homset.__init__(
            self,
            domain,
            codomain,
            category=domain.construction_category(),
        )

    def _element_constructor_(self, legs):
        if not isinstance(legs, (tuple, list)):
            raise TypeError(
                'an arrow morphism requires source and target legs'
            )
        if len(legs) != 2:
            raise ValueError(
                'an arrow morphism requires exactly two legs'
            )
        return self.from_legs(legs[0], legs[1])

    def from_legs(self, source_leg, target_leg):
        return ArrowMorphism(
            self,
            source_leg,
            target_leg,
        )

    def identity(self):
        if self.domain() is not self.codomain():
            raise TypeError(
                'the identity is defined only in an endomorphism homset'
            )
        source_identity = self.domain().source().Hom(
            self.domain().source()
        ).identity()
        target_identity = self.domain().target().Hom(
            self.domain().target()
        ).identity()
        return self.from_legs(
            source_identity,
            target_identity,
        )


class CosliceHomset(ArrowHomset):
    def _element_constructor_(self, target_leg):
        source_identity = self.domain().source().Hom(
            self.domain().source()
        ).identity()
        return self.from_legs(
            source_identity,
            target_leg,
        )

    def from_legs(self, source_leg, target_leg):
        identity = self.domain().source().Hom(
            self.domain().source()
        ).identity()
        status = _morphism_is_identity(source_leg)
        if status is None:
            status = self.domain().arrow_category().morphisms_equal(
                source_leg,
                identity,
            )
        if status is False:
            raise ValueError(
                'a coslice morphism must have identity source leg'
            )
        if status is None:
            raise NotImplementedError(
                'the source leg could not be certified as the identity'
            )
        return ArrowMorphism(
            self,
            source_leg,
            target_leg,
        )


class SliceHomset(ArrowHomset):
    def _element_constructor_(self, source_leg):
        target_identity = self.domain().target().Hom(
            self.domain().target()
        ).identity()
        return self.from_legs(
            source_leg,
            target_identity,
        )

    def from_legs(self, source_leg, target_leg):
        identity = self.domain().target().Hom(
            self.domain().target()
        ).identity()
        status = _morphism_is_identity(target_leg)
        if status is None:
            status = self.domain().arrow_category().morphisms_equal(
                target_leg,
                identity,
            )
        if status is False:
            raise ValueError(
                'a slice morphism must have identity target leg'
            )
        if status is None:
            raise NotImplementedError(
                'the target leg could not be certified as the identity'
            )
        return ArrowMorphism(
            self,
            source_leg,
            target_leg,
        )


class ArrowMorphism(Element):
    def __init__(
        self,
        parent,
        source_leg,
        target_leg,
        square_verified=False,
    ):
        source_arrow = parent.domain()
        target_arrow = parent.codomain()
        if source_leg.domain() is not source_arrow.source():
            raise ValueError(
                'the source leg has the wrong domain'
            )
        if source_leg.codomain() is not target_arrow.source():
            raise ValueError(
                'the source leg has the wrong codomain'
            )
        if target_leg.domain() is not source_arrow.target():
            raise ValueError(
                'the target leg has the wrong domain'
            )
        if target_leg.codomain() is not target_arrow.target():
            raise ValueError(
                'the target leg has the wrong codomain'
            )
        if not square_verified:
            status = source_arrow.arrow_category().square_status(
                source_arrow.arrow(),
                target_arrow.arrow(),
                source_leg,
                target_leg,
            )
            if status is False:
                raise ValueError(
                    'the arrow square does not commute'
                )
            if status is None:
                raise NotImplementedError(
                    'the arrow square could not be certified'
                )
        self._source_leg = source_leg
        self._target_leg = target_leg
        Element.__init__(self, parent)

    def domain(self):
        return self.parent().domain()

    def codomain(self):
        return self.parent().codomain()

    def source_leg(self):
        return self._source_leg

    def target_leg(self):
        return self._target_leg

    def underlying_morphism(self):
        if self.domain().arrow_kind() == 'coslice':
            return self._target_leg
        if self.domain().arrow_kind() == 'slice':
            return self._source_leg
        raise AttributeError(
            'a general arrow morphism has two equally primary legs'
        )

    def is_identity(self):
        if self.domain() is not self.codomain():
            return False
        arrow_category = self.domain().arrow_category()
        source_identity = self.domain().source().Hom(
            self.domain().source()
        ).identity()
        target_identity = self.domain().target().Hom(
            self.domain().target()
        ).identity()
        source_status = arrow_category.morphisms_equal(
            self._source_leg,
            source_identity,
        )
        target_status = arrow_category.morphisms_equal(
            self._target_leg,
            target_identity,
        )
        if source_status is False or target_status is False:
            return False
        if source_status is True and target_status is True:
            return True
        raise NotImplementedError(
            'identity of the arrow morphism could not be certified'
        )

    def _repr_defn(self):
        return (
            f'Source leg: {self._source_leg}\n'
            f'Target leg: {self._target_leg}'
        )

    def __mul__(self, right):
        if not isinstance(right, ArrowMorphism):
            return NotImplemented
        if right.codomain() is not self.domain():
            raise TypeError(
                'the arrow morphisms are not composable'
            )
        try:
            if right.is_identity():
                return self
        except NotImplementedError:
            pass
        try:
            if self.is_identity():
                return right
        except NotImplementedError:
            pass
        homset = Hom(
            right.domain(),
            self.codomain(),
        )
        return ArrowMorphism(
            homset,
            self._source_leg * right._source_leg,
            self._target_leg * right._target_leg,
            square_verified=True,
        )


print('Installed arrow, slice, and coslice functorial constructions.')

Installed arrow, slice, and coslice functorial constructions.


### Arrow-category regression

We construct two arrows $\eta_1,\eta_2:\mathbf Q[t]\to\mathbf Q[x]$, with $\eta_1(t)=x$ and $\eta_2(t)=x^2$. The square with target leg $x\mapsto x^2$ commutes. The square with target leg $\operatorname{id}_{\mathbf Q[x]}$ does not.

In [4]:
R_arrow_regression = PolynomialRing(
    QQ,
    names=('t_arrow_regression',),
)
t_arrow_regression = R_arrow_regression.gen()
A_arrow_regression = PolynomialRing(
    QQ,
    names=('x_arrow_regression',),
)
x_arrow_regression = A_arrow_regression.gen()

eta_linear_arrow_regression = R_arrow_regression.hom(
    (x_arrow_regression,),
    A_arrow_regression,
)
eta_square_arrow_regression = R_arrow_regression.hom(
    (x_arrow_regression**2,),
    A_arrow_regression,
)
valid_target_leg_arrow_regression = A_arrow_regression.hom(
    (x_arrow_regression**2,),
    A_arrow_regression,
)
invalid_target_leg_arrow_regression = A_arrow_regression.Hom(
    A_arrow_regression
).identity()
source_identity_arrow_regression = R_arrow_regression.Hom(
    R_arrow_regression
).identity()

ArrowRings_regression = CommutativeRings().ArrowCategory()
ArrowRings_regression.register_morphism_equality_certificate(
    PolynomialGeneratorEqualityCertificate(
        R_arrow_regression
    )
)
source_arrow_regression = ArrowRings_regression.arrow_object(
    eta_linear_arrow_regression
)
target_arrow_regression = ArrowRings_regression.arrow_object(
    eta_square_arrow_regression
)
valid_square_arrow_regression = Hom(
    source_arrow_regression,
    target_arrow_regression,
)(
    (
        source_identity_arrow_regression,
        valid_target_leg_arrow_regression,
    )
)

assert valid_square_arrow_regression.source_leg() == (
    source_identity_arrow_regression
)
assert valid_square_arrow_regression.target_leg() == (
    valid_target_leg_arrow_regression
)
assert ArrowRings_regression.square_status(
    eta_linear_arrow_regression,
    eta_square_arrow_regression,
    source_identity_arrow_regression,
    valid_target_leg_arrow_regression,
) is True

try:
    Hom(
        source_arrow_regression,
        target_arrow_regression,
    )(
        (
            source_identity_arrow_regression,
            invalid_target_leg_arrow_regression,
        )
    )
except ValueError as invalid_square_error_regression:
    assert 'does not commute' in str(
        invalid_square_error_regression
    )
else:
    raise AssertionError(
        'the arrow category accepted a noncommutative square'
    )

CosliceRings_regression = CommutativeRings().CosliceCategory(
    R_arrow_regression
)
coslice_source_regression = CosliceRings_regression.coslice_object(
    eta_linear_arrow_regression
)
coslice_target_regression = CosliceRings_regression.coslice_object(
    eta_square_arrow_regression
)
coslice_morphism_regression = Hom(
    coslice_source_regression,
    coslice_target_regression,
)(valid_target_leg_arrow_regression)

assert coslice_morphism_regression.source_leg() == (
    source_identity_arrow_regression
)
assert coslice_morphism_regression.target_leg() == (
    valid_target_leg_arrow_regression
)
assert CosliceRings_regression.arrow_category() is (
    ArrowRings_regression
)

print('Arrow-category regression passed.')
print('valid target leg =', valid_target_leg_arrow_regression)
print('invalid identity square rejected =', True)

Arrow-category regression passed.
valid target leg = Ring endomorphism of Univariate Polynomial Ring in x_arrow_regression over Rational Field
  Defn: x_arrow_regression |--> x_arrow_regression^2
invalid identity square rejected = True


## Routed coproducts in a coslice category

The coproduct in $(R\downarrow\mathbf{CRing})$ is a pushout of rings. The semantic operation is owned by the coslice category. Computational presentations only select a backend. The first backend implements

$$
(R[x_1,\ldots,x_m][z]/(z^n-f))\otimes_R R_g
\cong
R_g[x_1,\ldots,x_m][z]/(z^n-f_g),
$$

where the defining polynomial is monic in $z$.

In [5]:
class ConstructionBackendRegistry(SageObject):
    def __init__(self, construction_name):
        self._construction_name = construction_name
        self._entries = []

    def register(
        self,
        name,
        predicate,
        constructor,
        priority=0,
    ):
        self._entries.append(
            (priority, name, predicate, constructor)
        )
        self._entries.sort(
            key=lambda entry: entry[0],
            reverse=True,
        )
        return self

    def dispatch(self, *args, backend=None):
        predicate_failures = []
        for priority, name, predicate, constructor in self._entries:
            if backend is not None and name != backend:
                continue
            try:
                applies = bool(predicate(*args))
            except Exception as error:
                predicate_failures.append(
                    (
                        name,
                        type(error).__name__,
                        str(error),
                    )
                )
                continue
            if applies:
                return constructor(*args)
        if backend is not None:
            raise NotImplementedError(
                f'backend {backend!r} is unavailable for this input'
            )
        raise NotImplementedError(
            f'no {self._construction_name} backend applies; '
            f'predicate failures = {predicate_failures}'
        )


class PrincipalLocalizationEqualityCertificate(
    MorphismEqualityCertificate
):
    def __init__(
        self,
        localization_ring,
        base_structure,
        base_certificate,
    ):
        self._domain = localization_ring
        self._base_structure = base_structure
        self._base_certificate = base_certificate

    def domain(self):
        return self._domain

    def verify(self, left, right):
        if left.domain() is not self._domain:
            return False
        if right.domain() is not self._domain:
            return False
        if left.codomain() is not right.codomain():
            return False
        left_on_base = left * self._base_structure
        right_on_base = right * self._base_structure
        if _native_equality_proves_equal(
            left_on_base,
            right_on_base,
        ) is True:
            return True
        return self._base_certificate.verify(
            left_on_base,
            right_on_base,
        )


class MonicPolynomialAlgebraEqualityCertificate(
    MorphismEqualityCertificate
):
    def __init__(self, presentation, base_certificate):
        self._presentation = presentation
        self._domain = presentation.algebra()
        self._base_certificate = base_certificate

    def domain(self):
        return self._domain

    def verify(self, left, right):
        if left.domain() is not self._domain:
            return False
        if right.domain() is not self._domain:
            return False
        if left.codomain() is not right.codomain():
            return False
        if left(self._domain.gen()) != right(self._domain.gen()):
            return False

        affine_ring = self._presentation.affine_ring()
        if not all(
            left(self._domain(generator))
            == right(self._domain(generator))
            for generator in affine_ring.gens()
        ):
            return False

        left_on_base = (
            left
            * self._presentation.structure_morphism()
        )
        right_on_base = (
            right
            * self._presentation.structure_morphism()
        )
        if _native_equality_proves_equal(
            left_on_base,
            right_on_base,
        ) is True:
            return True
        return self._base_certificate.verify(
            left_on_base,
            right_on_base,
        )


class PrincipalLocalizationPresentation(SageObject):
    kind = 'principal_localization'

    def __init__(self, base_ring, element):
        self._base_ring = base_ring
        self._element = base_ring(element)
        self._ring = base_ring.localization(
            self._element
        )
        self._structure_morphism = base_ring.hom(
            tuple(
                self._ring(generator)
                for generator in base_ring.gens()
            ),
            self._ring,
        )
        assert self._ring(self._element).is_unit()

    def base_ring(self):
        return self._base_ring

    def element(self):
        return self._element

    def ring(self):
        return self._ring

    def structure_morphism(self):
        return self._structure_morphism

    def equality_certificate(self, base_certificate):
        return PrincipalLocalizationEqualityCertificate(
            self._ring,
            self._structure_morphism,
            base_certificate,
        )


class MonicPolynomialAlgebraPresentation(SageObject):
    kind = 'monic_polynomial_algebra'

    def __init__(
        self,
        base_ring,
        affine_variable_names,
        cover_variable_name,
        branch_polynomial,
        degree=2,
        quotient_name=None,
    ):
        self._base_ring = base_ring
        self._affine_variable_names = tuple(
            affine_variable_names
        )
        self._cover_variable_name = str(
            cover_variable_name
        )
        self._degree = Integer(degree)
        self._affine_ring = PolynomialRing(
            base_ring,
            len(self._affine_variable_names),
            names=self._affine_variable_names,
        )
        self._branch_polynomial = self._affine_ring(
            branch_polynomial
        )
        self._polynomial_ring = PolynomialRing(
            self._affine_ring,
            names=(self._cover_variable_name,),
        )
        cover_coordinate = self._polynomial_ring.gen()
        self._modulus = (
            cover_coordinate**self._degree
            - self._polynomial_ring(
                self._branch_polynomial
            )
        )
        if not self._modulus.is_monic():
            raise ValueError(
                'the cyclic-cover polynomial must be monic'
            )
        if quotient_name is None:
            quotient_name = (
                f'{self._cover_variable_name}bar'
            )
        self._algebra = self._polynomial_ring.quotient(
            self._modulus,
            names=(quotient_name,),
        )
        self._structure_morphism = (
            self._algebra.coerce_map_from(base_ring)
        )
        if self._structure_morphism is None:
            raise ValueError(
                'the algebra does not retain its base structure map'
            )

    def base_ring(self):
        return self._base_ring

    def affine_variable_names(self):
        return self._affine_variable_names

    def cover_variable_name(self):
        return self._cover_variable_name

    def degree(self):
        return self._degree

    def affine_ring(self):
        return self._affine_ring

    def branch_polynomial(self):
        return self._branch_polynomial

    def polynomial_ring(self):
        return self._polynomial_ring

    def modulus(self):
        return self._modulus

    def algebra(self):
        return self._algebra

    def structure_morphism(self):
        return self._structure_morphism

    def equality_certificate(self, base_certificate):
        return MonicPolynomialAlgebraEqualityCertificate(
            self,
            base_certificate,
        )


def _arrow_morphisms_equal(self, left, right):
    if _native_equality_proves_equal(left, right) is True:
        return True
    if left.domain() is not right.domain():
        return False
    if left.codomain() is not right.codomain():
        return False
    source_status = self.morphisms_equal(
        left.source_leg(),
        right.source_leg(),
    )
    target_status = self.morphisms_equal(
        left.target_leg(),
        right.target_leg(),
    )
    if source_status is False or target_status is False:
        return False
    if source_status is True and target_status is True:
        return True
    return None


ArrowCategoryConstruction.arrow_morphisms_equal = (
    _arrow_morphisms_equal
)


class CoproductDiagram(SageObject):
    def __init__(
        self,
        category,
        left,
        right,
        apex,
        left_injection,
        right_injection,
        universal_builder,
        certificate,
        backend,
    ):
        self._category = category
        self._left = left
        self._right = right
        self._apex = apex
        self._left_injection = left_injection
        self._right_injection = right_injection
        self._universal_builder = universal_builder
        self._certificate = certificate
        self._backend = backend
        if not self.commutes():
            raise ArithmeticError(
                'the coproduct injections do not commute with the base maps'
            )

    def left(self):
        return self._left

    def right(self):
        return self._right

    def apex(self):
        return self._apex

    def left_injection(self):
        return self._left_injection

    def right_injection(self):
        return self._right_injection

    def backend(self):
        return self._backend

    def certificate(self):
        return self._certificate

    def commutes(self):
        arrow_category = self._category.arrow_category()
        left_status = arrow_category.square_status(
            self._left.arrow(),
            self._apex.arrow(),
            self._left_injection.source_leg(),
            self._left_injection.target_leg(),
        )
        right_status = arrow_category.square_status(
            self._right.arrow(),
            self._apex.arrow(),
            self._right_injection.source_leg(),
            self._right_injection.target_leg(),
        )
        return left_status is True and right_status is True

    def universal_morphism(
        self,
        target,
        left_map,
        right_map,
    ):
        if left_map.domain() is not self._left:
            raise ValueError(
                'the left map has the wrong domain'
            )
        if left_map.codomain() is not target:
            raise ValueError(
                'the left map has the wrong codomain'
            )
        if right_map.domain() is not self._right:
            raise ValueError(
                'the right map has the wrong domain'
            )
        if right_map.codomain() is not target:
            raise ValueError(
                'the right map has the wrong codomain'
            )

        mediator = self._universal_builder(
            target,
            left_map,
            right_map,
        )
        arrow_category = self._category.arrow_category()
        left_composite = (
            mediator * self._left_injection
        )
        right_composite = (
            mediator * self._right_injection
        )
        left_status = arrow_category.arrow_morphisms_equal(
            left_composite,
            left_map,
        )
        right_status = arrow_category.arrow_morphisms_equal(
            right_composite,
            right_map,
        )
        if left_status is False:
            raise ArithmeticError(
                'the universal mediator fails on the left injection'
            )
        if right_status is False:
            raise ArithmeticError(
                'the universal mediator fails on the right injection'
            )
        if left_status is None or right_status is None:
            raise NotImplementedError(
                'the universal mediator equations could not be certified'
            )
        return mediator


_COSLICE_COPRODUCT_BACKENDS = ConstructionBackendRegistry(
    'coslice coproduct'
)


def _coslice_coproduct(
    self,
    left,
    right,
    backend=None,
):
    if left.construction_category() is not self:
        raise TypeError(
            'the left object is not in this coslice category'
        )
    if right.construction_category() is not self:
        raise TypeError(
            'the right object is not in this coslice category'
        )
    return _COSLICE_COPRODUCT_BACKENDS.dispatch(
        self,
        left,
        right,
        backend=backend,
    )


CosliceCategoryConstruction.coproduct = _coslice_coproduct


def _monic_localization_backend_applies(
    category,
    left,
    right,
):
    left_monic = left.presentation(
        'monic_polynomial_algebra'
    )
    right_monic = right.presentation(
        'monic_polynomial_algebra'
    )
    left_local = left.presentation(
        'principal_localization'
    )
    right_local = right.presentation(
        'principal_localization'
    )
    return bool(
        (left_monic is not None and right_local is not None)
        or
        (right_monic is not None and left_local is not None)
    )


def _monic_localization_backend(
    category,
    left,
    right,
):
    swapped = False
    if left.presentation(
        'monic_polynomial_algebra'
    ) is None:
        left, right = right, left
        swapped = True

    monic_presentation = left.presentation(
        'monic_polynomial_algebra'
    )
    localization_presentation = right.presentation(
        'principal_localization'
    )
    base_ring = category.base_object()
    if monic_presentation.base_ring() is not base_ring:
        raise ValueError(
            'the monic algebra has the wrong base ring'
        )
    if localization_presentation.base_ring() is not base_ring:
        raise ValueError(
            'the localization has the wrong base ring'
        )

    arrow_category = category.arrow_category()
    base_certificate = arrow_category._morphism_equality_certificates.get(
        id(base_ring)
    )
    if base_certificate is None:
        raise NotImplementedError(
            'the base-ring morphism equality has not been certified'
        )

    localized_ring = localization_presentation.ring()
    localized_structure = (
        localization_presentation.structure_morphism()
    )
    localization_certificate = (
        localization_presentation.equality_certificate(
            base_certificate
        )
    )
    arrow_category.register_morphism_equality_certificate(
        localization_certificate
    )

    affine_ring = monic_presentation.affine_ring()
    localized_affine_ring = PolynomialRing(
        localized_ring,
        len(monic_presentation.affine_variable_names()),
        names=monic_presentation.affine_variable_names(),
    )
    affine_base_change = affine_ring.hom(
        tuple(localized_affine_ring.gens()),
        localized_affine_ring,
        base_map=localized_structure,
    )
    localized_branch = affine_base_change(
        monic_presentation.branch_polynomial()
    )
    localized_monic_presentation = (
        MonicPolynomialAlgebraPresentation(
            localized_ring,
            monic_presentation.affine_variable_names(),
            monic_presentation.cover_variable_name(),
            localized_branch,
            degree=monic_presentation.degree(),
            quotient_name=(
                f'{monic_presentation.cover_variable_name()}bar_bc'
            ),
        )
    )

    source_algebra = monic_presentation.algebra()
    localized_algebra = (
        localized_monic_presentation.algebra()
    )
    localized_algebra_structure = (
        localized_monic_presentation.structure_morphism()
    )
    apex_structure = (
        localized_algebra_structure
        * localized_structure
    )
    apex = category.coslice_object(apex_structure)
    apex.register_presentation(
        localized_monic_presentation
    )

    source_certificate = (
        monic_presentation.equality_certificate(
            base_certificate
        )
    )
    localized_algebra_certificate = (
        localized_monic_presentation.equality_certificate(
            localization_certificate
        )
    )
    arrow_category.register_morphism_equality_certificate(
        source_certificate
    )
    arrow_category.register_morphism_equality_certificate(
        localized_algebra_certificate
    )

    source_affine_to_apex = affine_ring.hom(
        tuple(
            localized_algebra(
                localized_affine_ring.gen(index)
            )
            for index in range(
                localized_affine_ring.ngens()
            )
        ),
        localized_algebra,
        base_map=apex_structure,
    )
    source_injection_ring_map = source_algebra.hom(
        (localized_algebra.gen(),),
        localized_algebra,
        base_map=source_affine_to_apex,
    )
    localization_injection_ring_map = (
        localized_algebra_structure
    )

    left_injection = Hom(
        left,
        apex,
    )(source_injection_ring_map)
    right_injection = Hom(
        right,
        apex,
    )(localization_injection_ring_map)

    def universal_builder(
        target,
        left_map,
        right_map,
    ):
        target_ring = target.underlying_object()
        left_ring_map = left_map.target_leg()
        right_ring_map = right_map.target_leg()
        localized_affine_to_target = (
            localized_affine_ring.hom(
                tuple(
                    left_ring_map(
                        source_algebra(generator)
                    )
                    for generator in affine_ring.gens()
                ),
                target_ring,
                base_map=right_ring_map,
            )
        )
        mediator_ring_map = localized_algebra.hom(
            (
                left_ring_map(
                    source_algebra.gen()
                ),
            ),
            target_ring,
            base_map=localized_affine_to_target,
        )
        return Hom(
            apex,
            target,
        )(mediator_ring_map)

    diagram = CoproductDiagram(
        category,
        left,
        right,
        apex,
        left_injection,
        right_injection,
        universal_builder,
        certificate=(
            'base change of a monic polynomial algebra '
            'along a principal localization'
        ),
        backend=(
            'monic_polynomial_principal_localization'
        ),
    )

    if not swapped:
        return diagram

    return CoproductDiagram(
        category,
        right,
        left,
        apex,
        right_injection,
        left_injection,
        lambda target, first_map, second_map: (
            universal_builder(
                target,
                second_map,
                first_map,
            )
        ),
        certificate=diagram.certificate(),
        backend=diagram.backend(),
    )


_COSLICE_COPRODUCT_BACKENDS.register(
    'monic_polynomial_principal_localization',
    _monic_localization_backend_applies,
    _monic_localization_backend,
    priority=100,
)

print('Installed routed coproducts in ring coslices.')

Installed routed coproducts in ring coslices.


### Coproduct regression

We compute the coproduct of the $R$-algebras

$$
R[u][z]/(z^2-u-a)
\qquad\text{and}\qquad
R_a,
$$

for $R=\mathbf Q[a]$, and verify the canonical maps and the universal mediator.

In [6]:
R_coproduct_regression = PolynomialRing(
    QQ,
    names=('a_coproduct_regression',),
)
a_coproduct_regression = R_coproduct_regression.gen()
affine_ring_coproduct_regression = PolynomialRing(
    R_coproduct_regression,
    1,
    names=('u_coproduct_regression',),
)
u_coproduct_regression = (
    affine_ring_coproduct_regression.gen()
)

monic_presentation_coproduct_regression = (
    MonicPolynomialAlgebraPresentation(
        R_coproduct_regression,
        ('u_coproduct_regression',),
        'z_coproduct_regression',
        u_coproduct_regression
        + a_coproduct_regression,
        degree=2,
    )
)
localization_presentation_coproduct_regression = (
    PrincipalLocalizationPresentation(
        R_coproduct_regression,
        a_coproduct_regression,
    )
)

Coslice_coproduct_regression = (
    CommutativeRings().CosliceCategory(
        R_coproduct_regression
    )
)
Coslice_coproduct_regression.register_morphism_equality_certificate(
    PolynomialGeneratorEqualityCertificate(
        R_coproduct_regression
    )
)
monic_object_coproduct_regression = (
    Coslice_coproduct_regression.coslice_object(
        monic_presentation_coproduct_regression
        .structure_morphism()
    ).register_presentation(
        monic_presentation_coproduct_regression
    )
)
localization_object_coproduct_regression = (
    Coslice_coproduct_regression.coslice_object(
        localization_presentation_coproduct_regression
        .structure_morphism()
    ).register_presentation(
        localization_presentation_coproduct_regression
    )
)

coproduct_regression = (
    Coslice_coproduct_regression.coproduct(
        monic_object_coproduct_regression,
        localization_object_coproduct_regression,
    )
)

assert coproduct_regression.commutes()
assert coproduct_regression.backend() == (
    'monic_polynomial_principal_localization'
)
mediator_coproduct_regression = (
    coproduct_regression.universal_morphism(
        coproduct_regression.apex(),
        coproduct_regression.left_injection(),
        coproduct_regression.right_injection(),
    )
)
identity_coproduct_regression = (
    Hom(
        coproduct_regression.apex(),
        coproduct_regression.apex(),
    ).identity()
)
assert (
    coproduct_regression.apex()
    .category()
    .arrow_category()
    .arrow_morphisms_equal(
        mediator_coproduct_regression,
        identity_coproduct_regression,
    )
    is True
)

print('Coslice coproduct regression passed.')
print('backend =', coproduct_regression.backend())
print('apex =', coproduct_regression.apex().underlying_object())
print('certificate =', coproduct_regression.certificate())

Coslice coproduct regression passed.
backend = monic_polynomial_principal_localization
apex = Univariate Quotient Polynomial Ring in z_coproduct_regressionbar_bc over Multivariate Polynomial Ring in u_coproduct_regression over Univariate Polynomial Ring in a_coproduct_regression over Rational Field localized at (a_coproduct_regression,) with modulus z_coproduct_regression^2 - u_coproduct_regression - a_coproduct_regression
certificate = base change of a monic polynomial algebra along a principal localization


## `Spec` and pullbacks of affine schemes

The native contravariant construction `Spec` sends commutative rings and ring morphisms to affine schemes and morphisms of affine schemes. We extend the same construction to ring-coslice objects and morphisms. Applying `Spec` to a certified ring pushout produces the corresponding pullback in the slice category of schemes.

In [7]:
def _coordinate_ring_morphism(morphism):
    if hasattr(morphism, 'ring_homomorphism'):
        return morphism.ring_homomorphism()
    if isinstance(morphism, FormalCompositeMap):
        first_ring_map = _coordinate_ring_morphism(
            morphism.first()
        )
        remaining_ring_map = _coordinate_ring_morphism(
            morphism.then()
        )
        return first_ring_map * remaining_ring_map
    if _morphism_is_identity(morphism) is True:
        coordinate_ring = (
            morphism.domain().coordinate_ring()
        )
        return coordinate_ring.Hom(
            coordinate_ring
        ).identity()
    raise NotImplementedError(
        'the morphism of affine schemes has no accessible coordinate-ring morphism'
    )


class MorphismOfAffineSchemesEqualityCertificate(
    MorphismEqualityCertificate
):
    def __init__(
        self,
        affine_scheme,
        ring_arrow_category,
    ):
        self._domain = affine_scheme
        self._ring_arrow_category = ring_arrow_category

    def domain(self):
        return self._domain

    def verify(self, left, right):
        if left.domain() is not self._domain:
            return False
        if right.domain() is not self._domain:
            return False
        if left.codomain() is not right.codomain():
            return False
        left_ring_map = _coordinate_ring_morphism(
            left
        )
        right_ring_map = _coordinate_ring_morphism(
            right
        )
        return self._ring_arrow_category.morphisms_equal(
            left_ring_map,
            right_ring_map,
        )


class AffineSchemeSquareCertificate(
    SquareCommutativityCertificate
):
    def __init__(self, ring_arrow_category):
        self._ring_arrow_category = ring_arrow_category

    def applies(
        self,
        source_arrow,
        target_arrow,
        source_leg,
        target_leg,
    ):
        for morphism in (
            source_arrow,
            target_arrow,
            source_leg,
            target_leg,
        ):
            try:
                _coordinate_ring_morphism(morphism)
            except NotImplementedError:
                return False
        return True

    def verify(
        self,
        source_arrow,
        target_arrow,
        source_leg,
        target_leg,
    ):
        source_structure = _coordinate_ring_morphism(
            source_arrow
        )
        target_structure = _coordinate_ring_morphism(
            target_arrow
        )
        source_leg_ring_map = _coordinate_ring_morphism(
            source_leg
        )
        target_leg_ring_map = _coordinate_ring_morphism(
            target_leg
        )
        left_ring_composite = (
            source_structure
            * target_leg_ring_map
        )
        right_ring_composite = (
            source_leg_ring_map
            * target_structure
        )
        return self._ring_arrow_category.morphisms_equal(
            left_ring_composite,
            right_ring_composite,
        )


_SPEC_SLICE_CATEGORIES = {}
_SPEC_OBJECT_CACHE = {}


def _scheme_slice_for_ring_coslice(
    ring_coslice_category,
):
    if not hasattr(
        ring_coslice_category,
        'base_object',
    ):
        raise TypeError(
            'Spec requires a coslice category of commutative rings'
        )
    if not ring_coslice_category.base_category().is_subcategory(
        CommutativeRings()
    ):
        raise TypeError(
            'Spec requires a coslice category of commutative rings'
        )
    cache_key = id(ring_coslice_category)
    if cache_key not in _SPEC_SLICE_CATEGORIES:
        base_scheme = _sage_Spec(
            ring_coslice_category.base_object()
        )
        slice_category = Schemes().SliceCategory(
            base_scheme
        )
        slice_category.register_square_certificate(
            AffineSchemeSquareCertificate(
                ring_coslice_category.arrow_category()
            )
        )
        _SPEC_SLICE_CATEGORIES[
            cache_key
        ] = slice_category
    return _SPEC_SLICE_CATEGORIES[cache_key]


def _spec_of_ring_morphism(ring_morphism):
    if not hasattr(ring_morphism, 'domain'):
        raise TypeError(
            'Spec on a morphism requires a categorical domain'
        )
    if not hasattr(ring_morphism, 'codomain'):
        raise TypeError(
            'Spec on a morphism requires a categorical codomain'
        )
    if ring_morphism.domain() not in CommutativeRings():
        raise TypeError(
            'the source of the ring morphism is not a commutative ring'
        )
    if ring_morphism.codomain() not in CommutativeRings():
        raise TypeError(
            'the target of the ring morphism is not a commutative ring'
        )
    source_scheme = _sage_Spec(
        ring_morphism.codomain()
    )
    target_scheme = _sage_Spec(
        ring_morphism.domain()
    )
    return source_scheme.hom(
        ring_morphism,
        target_scheme,
    )


def _spec_of_coslice_object(ring_object):
    if ring_object.arrow_kind() != 'coslice':
        raise TypeError(
            'Spec on an arrow object currently requires a ring-coslice object'
        )
    ring_coslice_category = (
        ring_object.construction_category()
    )
    cache_key = id(ring_object)
    if cache_key not in _SPEC_OBJECT_CACHE:
        scheme_slice = _scheme_slice_for_ring_coslice(
            ring_coslice_category
        )
        structure_morphism = Spec(
            ring_object.arrow()
        )
        scheme_object = scheme_slice.slice_object(
            structure_morphism
        )
        scheme_object._source_coslice_object = (
            ring_object
        )
        scheme_slice.arrow_category().register_morphism_equality_certificate(
            MorphismOfAffineSchemesEqualityCertificate(
                structure_morphism.domain(),
                ring_coslice_category.arrow_category(),
            )
        )
        _SPEC_OBJECT_CACHE[cache_key] = scheme_object
    return _SPEC_OBJECT_CACHE[cache_key]


def _spec_of_coslice_morphism(ring_morphism):
    if ring_morphism.domain().arrow_kind() != 'coslice':
        raise TypeError(
            'Spec on an arrow morphism currently requires a ring-coslice morphism'
        )
    source_scheme_object = Spec(
        ring_morphism.codomain()
    )
    target_scheme_object = Spec(
        ring_morphism.domain()
    )
    scheme_morphism = Spec(
        ring_morphism.target_leg()
    )
    return Hom(
        source_scheme_object,
        target_scheme_object,
    )(scheme_morphism)


class PullbackDiagram(SageObject):
    def __init__(self, source_pushout):
        self._source_pushout = source_pushout
        self._left = Spec(
            source_pushout.left()
        )
        self._right = Spec(
            source_pushout.right()
        )
        self._apex = Spec(
            source_pushout.apex()
        )
        self._category = (
            self._apex.construction_category()
        )
        self._projection_to_left = Spec(
            source_pushout.left_injection()
        )
        self._projection_to_right = Spec(
            source_pushout.right_injection()
        )
        if not self.commutes():
            raise ArithmeticError(
                'the pullback square of affine schemes does not commute'
            )

    def category(self):
        return self._category

    def left(self):
        return self._left

    def right(self):
        return self._right

    def apex(self):
        return self._apex

    def projection_to_left(self):
        return self._projection_to_left

    def projection_to_right(self):
        return self._projection_to_right

    def source_pushout(self):
        return self._source_pushout

    def certificate(self):
        return (
            'Spec applied to: '
            f'{self._source_pushout.certificate()}'
        )

    def commutes(self):
        arrow_category = (
            self._category.arrow_category()
        )
        left_status = arrow_category.square_status(
            self._apex.arrow(),
            self._left.arrow(),
            self._projection_to_left.source_leg(),
            self._projection_to_left.target_leg(),
        )
        right_status = arrow_category.square_status(
            self._apex.arrow(),
            self._right.arrow(),
            self._projection_to_right.source_leg(),
            self._projection_to_right.target_leg(),
        )
        return (
            left_status is True
            and right_status is True
        )

    def universal_morphism(
        self,
        source,
        map_to_left,
        map_to_right,
    ):
        if not hasattr(
            source,
            '_source_coslice_object',
        ):
            raise NotImplementedError(
                'the source must currently lie in the essential image of Spec'
            )
        source_ring_object = (
            source._source_coslice_object
        )
        left_ring_map = Hom(
            self._source_pushout.left(),
            source_ring_object,
        )(
            _coordinate_ring_morphism(
                map_to_left.source_leg()
            )
        )
        right_ring_map = Hom(
            self._source_pushout.right(),
            source_ring_object,
        )(
            _coordinate_ring_morphism(
                map_to_right.source_leg()
            )
        )
        mediator_ring_map = (
            self._source_pushout
            .universal_morphism(
                source_ring_object,
                left_ring_map,
                right_ring_map,
            )
        )
        return Spec(mediator_ring_map)


def Spec(value, base_ring=None):
    if base_ring is not None:
        return _sage_Spec(
            value,
            base_ring,
        )
    if isinstance(value, CoproductDiagram):
        return PullbackDiagram(value)
    if isinstance(value, ArrowMorphism):
        return _spec_of_coslice_morphism(value)
    if isinstance(value, ArrowObject):
        return _spec_of_coslice_object(value)
    if (
        hasattr(value, 'domain')
        and hasattr(value, 'codomain')
        and value.domain() in CommutativeRings()
        and value.codomain() in CommutativeRings()
    ):
        return _spec_of_ring_morphism(value)
    return _sage_Spec(value)


print('Extended the native Spec construction to ring coslices and their coproducts.')

Extended the native Spec construction to ring coslices and their coproducts.


### Affine pullback regression

We apply affine `Spec` to the preceding ring coproduct. We verify the pullback square and recover the identity map from the pullback universal property.

In [8]:
pullback_regression = Spec(
    coproduct_regression
)
assert pullback_regression.commutes()
mediator_pullback_regression = (
    pullback_regression.universal_morphism(
        pullback_regression.apex(),
        pullback_regression.projection_to_left(),
        pullback_regression.projection_to_right(),
    )
)
identity_pullback_regression = Hom(
    pullback_regression.apex(),
    pullback_regression.apex(),
).identity()
status_pullback_regression = (
    pullback_regression.apex()
    .arrow_category()
    .arrow_morphisms_equal(
        mediator_pullback_regression,
        identity_pullback_regression,
    )
)
assert status_pullback_regression is True

print('Pullback regression under Spec passed.')
print('commutes =', pullback_regression.commutes())
print(
    'universal mediator equals identity =',
    status_pullback_regression,
)
print('apex =', pullback_regression.apex().underlying_object())

Pullback regression under Spec passed.
commutes = True
universal mediator equals identity = True
apex = Spectrum of Univariate Quotient Polynomial Ring in z_coproduct_regressionbar_bc over Multivariate Polynomial Ring in u_coproduct_regression over Univariate Polynomial Ring in a_coproduct_regression over Rational Field localized at (a_coproduct_regression,) with modulus z_coproduct_regression^2 - u_coproduct_regression - a_coproduct_regression


## Research vertical slice: a universal K3-cover chart over the exact free open

We construct the full $13$-parameter invariant coefficient ring, the affine $(0,0)$ chart of the universal branch equation, and the degree-two cover algebra. We form its coproduct with the principal localization at $a_{00}a_{04}a_{40}a_{44}$, apply affine `Spec`, and verify the deck and lifted sign involutions on the pulled-back algebra.

In [9]:
parameter_names_v2 = (
    'a00',
    'a02',
    'a04',
    'a11',
    'a13',
    'a20',
    'a22',
    'a24',
    'a31',
    'a33',
    'a40',
    'a42',
    'a44',
)
ParameterRing_v2 = PolynomialRing(
    QQ,
    len(parameter_names_v2),
    names=parameter_names_v2,
)
parameter_coordinates_v2 = (
    ParameterRing_v2.gens_dict()
)
AffineChartRing_v2 = PolynomialRing(
    ParameterRing_v2,
    2,
    names=('x0', 'x1'),
)
x0_v2, x1_v2 = AffineChartRing_v2.gens()

branch_function_00_v2 = (
    parameter_coordinates_v2['a44']
    * x0_v2**4
    * x1_v2**4
    + parameter_coordinates_v2['a42']
    * x0_v2**4
    * x1_v2**2
    + parameter_coordinates_v2['a33']
    * x0_v2**3
    * x1_v2**3
    + parameter_coordinates_v2['a24']
    * x0_v2**2
    * x1_v2**4
    + parameter_coordinates_v2['a40']
    * x0_v2**4
    + parameter_coordinates_v2['a31']
    * x0_v2**3
    * x1_v2
    + parameter_coordinates_v2['a22']
    * x0_v2**2
    * x1_v2**2
    + parameter_coordinates_v2['a13']
    * x0_v2
    * x1_v2**3
    + parameter_coordinates_v2['a04']
    * x1_v2**4
    + parameter_coordinates_v2['a20']
    * x0_v2**2
    + parameter_coordinates_v2['a11']
    * x0_v2
    * x1_v2
    + parameter_coordinates_v2['a02']
    * x1_v2**2
    + parameter_coordinates_v2['a00']
)
free_parameter_polynomial_v2 = (
    parameter_coordinates_v2['a00']
    * parameter_coordinates_v2['a04']
    * parameter_coordinates_v2['a40']
    * parameter_coordinates_v2['a44']
)

universal_chart_presentation_v2 = (
    MonicPolynomialAlgebraPresentation(
        ParameterRing_v2,
        ('x0', 'x1'),
        'z00',
        branch_function_00_v2,
        degree=2,
        quotient_name='zbar00',
    )
)
free_open_presentation_v2 = (
    PrincipalLocalizationPresentation(
        ParameterRing_v2,
        free_parameter_polynomial_v2,
    )
)

ParameterCoslice_v2 = (
    CommutativeRings().CosliceCategory(
        ParameterRing_v2
    )
)
ParameterCoslice_v2.register_morphism_equality_certificate(
    PolynomialGeneratorEqualityCertificate(
        ParameterRing_v2
    )
)
universal_chart_object_v2 = (
    ParameterCoslice_v2.coslice_object(
        universal_chart_presentation_v2
        .structure_morphism()
    ).register_presentation(
        universal_chart_presentation_v2
    )
)
free_open_object_v2 = (
    ParameterCoslice_v2.coslice_object(
        free_open_presentation_v2
        .structure_morphism()
    ).register_presentation(
        free_open_presentation_v2
    )
)

free_chart_pushout_v2 = (
    ParameterCoslice_v2.coproduct(
        universal_chart_object_v2,
        free_open_object_v2,
    )
)
localized_chart_presentation_v2 = (
    free_chart_pushout_v2
    .apex()
    .presentation(
        'monic_polynomial_algebra'
    )
)
LocalizedParameterRing_v2 = (
    free_open_presentation_v2.ring()
)
LocalizedCoverAlgebra_v2 = (
    localized_chart_presentation_v2.algebra()
)
LocalizedAffineChartRing_v2 = (
    localized_chart_presentation_v2.affine_ring()
)
x0_local_v2, x1_local_v2 = (
    LocalizedAffineChartRing_v2.gens()
)

fixed_origin_branch_value_v2 = (
    localized_chart_presentation_v2
    .branch_polynomial()
    .subs(
        {
            x0_local_v2: 0,
            x1_local_v2: 0,
        }
    )
)
free_coordinate_units_v2 = tuple(
    LocalizedParameterRing_v2(
        parameter_coordinates_v2[name]
    ).is_unit()
    for name in (
        'a00',
        'a04',
        'a40',
        'a44',
    )
)

assert len(ParameterRing_v2.gens()) == 13
assert free_chart_pushout_v2.commutes()
assert free_chart_pushout_v2.backend() == (
    'monic_polynomial_principal_localization'
)
assert LocalizedParameterRing_v2(
    free_parameter_polynomial_v2
).is_unit()
assert all(free_coordinate_units_v2)
assert fixed_origin_branch_value_v2 == (
    LocalizedParameterRing_v2(
        parameter_coordinates_v2['a00']
    )
)
assert fixed_origin_branch_value_v2.is_unit()
assert localized_chart_presentation_v2.modulus().is_monic()
assert localized_chart_presentation_v2.modulus().degree() == 2

identity_affine_chart_v2 = (
    LocalizedAffineChartRing_v2.Hom(
        LocalizedAffineChartRing_v2
    ).identity()
)
deck_involution_v2 = LocalizedCoverAlgebra_v2.hom(
    (-LocalizedCoverAlgebra_v2.gen(),),
    LocalizedCoverAlgebra_v2,
    base_map=identity_affine_chart_v2,
)
base_sign_involution_v2 = (
    LocalizedAffineChartRing_v2.hom(
        (
            -x0_local_v2,
            -x1_local_v2,
        ),
        LocalizedAffineChartRing_v2,
        base_map=(
            LocalizedParameterRing_v2.Hom(
                LocalizedParameterRing_v2
            ).identity()
        ),
    )
)
lifted_sign_involution_v2 = (
    LocalizedCoverAlgebra_v2.hom(
        (-LocalizedCoverAlgebra_v2.gen(),),
        LocalizedCoverAlgebra_v2,
        base_map=base_sign_involution_v2,
    )
)
identity_cover_algebra_v2 = (
    LocalizedCoverAlgebra_v2.Hom(
        LocalizedCoverAlgebra_v2
    ).identity()
)
localized_cover_equality_certificate_v2 = (
    ParameterCoslice_v2
    .arrow_category()
    ._morphism_equality_certificates[
        id(LocalizedCoverAlgebra_v2)
    ]
)

deck_involution_order_two_v2 = (
    localized_cover_equality_certificate_v2.verify(
        deck_involution_v2 * deck_involution_v2,
        identity_cover_algebra_v2,
    )
)
lifted_sign_order_two_v2 = (
    localized_cover_equality_certificate_v2.verify(
        lifted_sign_involution_v2
        * lifted_sign_involution_v2,
        identity_cover_algebra_v2,
    )
)
deck_preserves_parameter_structure_v2 = (
    ParameterCoslice_v2
    .arrow_category()
    .morphisms_equal(
        deck_involution_v2
        * free_chart_pushout_v2.apex().arrow(),
        free_chart_pushout_v2.apex().arrow(),
    )
)
lifted_sign_preserves_parameter_structure_v2 = (
    ParameterCoslice_v2
    .arrow_category()
    .morphisms_equal(
        lifted_sign_involution_v2
        * free_chart_pushout_v2.apex().arrow(),
        free_chart_pushout_v2.apex().arrow(),
    )
)

assert deck_involution_order_two_v2 is True
assert lifted_sign_order_two_v2 is True
assert deck_preserves_parameter_structure_v2 is True
assert lifted_sign_preserves_parameter_structure_v2 is True

free_chart_pullback_v2 = Spec(
    free_chart_pushout_v2
)
assert free_chart_pullback_v2.commutes()

show(
    LatexExpr(r'f_{00}='),
    branch_function_00_v2,
)
show(
    LatexExpr(r'g_{\mathrm{free}}='),
    free_parameter_polynomial_v2,
)
print('parameter count =', len(ParameterRing_v2.gens()))
print('free coordinate units =', free_coordinate_units_v2)
print(
    'fixed-origin branch value =',
    fixed_origin_branch_value_v2,
)
print(
    'fixed-origin branch value is a unit =',
    fixed_origin_branch_value_v2.is_unit(),
)
print('cover rank certificate =', localized_chart_presentation_v2.modulus().degree())
print('deck involution has order two =', deck_involution_order_two_v2)
print('lifted sign involution has order two =', lifted_sign_order_two_v2)
print('deck preserves parameter structure =', deck_preserves_parameter_structure_v2)
print('lifted sign preserves parameter structure =', lifted_sign_preserves_parameter_structure_v2)
print('affine pullback square commutes =', free_chart_pullback_v2.commutes())
print('pullback apex =', free_chart_pullback_v2.apex().underlying_object())

f_{00}= a44*x0^4*x1^4 + a42*x0^4*x1^2 + a33*x0^3*x1^3 + a24*x0^2*x1^4 + a40*x0^4 + a31*x0^3*x1 + a22*x0^2*x1^2 + a13*x0*x1^3 + a04*x1^4 + a20*x0^2 + a11*x0*x1 + a02*x1^2 + a00

g_{\mathrm{free}}= a00*a04*a40*a44

parameter count = 13
free coordinate units = (True, True, True, True)
fixed-origin branch value = a00
fixed-origin branch value is a unit = True
cover rank certificate = 2
deck involution has order two = True
lifted sign involution has order two = True
deck preserves parameter structure = True
lifted sign preserves parameter structure = True
affine pullback square commutes = True
pullback apex = Spectrum of Univariate Quotient Polynomial Ring in z00bar_bc over Multivariate Polynomial Ring in x0, x1 over Multivariate Polynomial Ring in a00, a02, a04, a11, a13, a20, a22, a24, a31, a33, a40, a42, a44 over Rational Field localized at (a44, a40, a04, a00) with modulus z00^2 + (-a44)*x0^4*x1^4 + (-a42)*x0^4*x1^2 + (-a33)*x0^3*x1^3 + (-a24)*x0^2*x1^4 + (-a40)*x0^4 + (-a31)*x0^3*x1 + (-a22)*x0^2*x1^2 + (-a13)*x0*x1^3 + (-a04)*x1^4 + (-a20)*x0^2 + (-a11)*x0*x1 + (-a02)*x1^2 - a00


## Finite indexing categories

We construct finite small categories from complete finite Hom-sets and a complete composition table. We verify identities and associativity exhaustively. Standard finite shapes are supplied for discrete diagrams, arrows, spans, cospans, and parallel pairs.

In [10]:
class FiniteCategoryMorphism(SageObject):
    def __init__(
        self,
        category,
        name,
        source,
        target,
        is_identity=False,
    ):
        self._category = category
        self._name = name
        self._source = source
        self._target = target
        self._is_identity = bool(is_identity)

    def category(self):
        return self._category

    def name(self):
        return self._name

    def source(self):
        return self._source

    def target(self):
        return self._target

    def is_identity(self):
        return self._is_identity

    def __hash__(self):
        return id(self)

    def __eq__(self, other):
        return self is other

    def _repr_(self):
        if self._is_identity:
            return f'id_{self._source}'
        return str(self._name)


class FiniteSmallCategory(SageObject):
    def __init__(
        self,
        objects,
        morphisms=(),
        compositions=None,
        name=None,
    ):
        self._objects = tuple(objects)
        if len(set(self._objects)) != len(self._objects):
            raise ValueError(
                'the object labels must be distinct'
            )
        self._name = name or 'finite category'
        self._morphisms_by_name = {}
        self._identities = {}

        for obj in self._objects:
            identity_name = ('identity', obj)
            identity = FiniteCategoryMorphism(
                self,
                identity_name,
                obj,
                obj,
                is_identity=True,
            )
            self._identities[obj] = identity
            self._morphisms_by_name[
                identity_name
            ] = identity

        for morphism_data in morphisms:
            if len(morphism_data) != 3:
                raise ValueError(
                    'each nonidentity morphism requires name, source, and target'
                )
            morphism_name, source, target = morphism_data
            if source not in self._objects:
                raise ValueError(
                    'a morphism source is not an object of the category'
                )
            if target not in self._objects:
                raise ValueError(
                    'a morphism target is not an object of the category'
                )
            if morphism_name in self._morphisms_by_name:
                raise ValueError(
                    'morphism names must be distinct'
                )
            self._morphisms_by_name[
                morphism_name
            ] = FiniteCategoryMorphism(
                self,
                morphism_name,
                source,
                target,
            )

        self._composition_table = {}
        supplied_compositions = compositions or {}
        for right in self.morphisms():
            for left in self.morphisms():
                if right.target() != left.source():
                    continue
                pair = (left, right)
                if left.is_identity():
                    composite = right
                elif right.is_identity():
                    composite = left
                else:
                    composition_key = (
                        left.name(),
                        right.name(),
                    )
                    if composition_key not in supplied_compositions:
                        raise ValueError(
                            'the composition table is incomplete for '
                            f'{left} after {right}'
                        )
                    result_name = supplied_compositions[
                        composition_key
                    ]
                    if result_name not in self._morphisms_by_name:
                        raise ValueError(
                            'a supplied composite is not a morphism of the category'
                        )
                    composite = self._morphisms_by_name[
                        result_name
                    ]
                if composite.source() != right.source():
                    raise ValueError(
                        'a composite has the wrong source'
                    )
                if composite.target() != left.target():
                    raise ValueError(
                        'a composite has the wrong target'
                    )
                self._composition_table[pair] = composite

        self._verify_associativity()

    @classmethod
    def discrete(cls, objects, name=None):
        return cls(
            objects,
            name=name or 'finite discrete category',
        )

    @classmethod
    def walking_arrow(cls):
        return cls(
            ('source', 'target'),
            morphisms=(
                ('arrow', 'source', 'target'),
            ),
            name='walking arrow',
        )

    @classmethod
    def walking_span(cls):
        return cls(
            ('center', 'left', 'right'),
            morphisms=(
                ('to_left', 'center', 'left'),
                ('to_right', 'center', 'right'),
            ),
            name='walking span',
        )

    @classmethod
    def walking_cospan(cls):
        return cls(
            ('left', 'right', 'center'),
            morphisms=(
                ('from_left', 'left', 'center'),
                ('from_right', 'right', 'center'),
            ),
            name='walking cospan',
        )

    @classmethod
    def parallel_pair(cls):
        return cls(
            ('source', 'target'),
            morphisms=(
                ('first', 'source', 'target'),
                ('second', 'source', 'target'),
            ),
            name='walking parallel pair',
        )

    def __hash__(self):
        return id(self)

    def __eq__(self, other):
        return self is other

    def _verify_associativity(self):
        for first in self.morphisms():
            for second in self.morphisms():
                if first.target() != second.source():
                    continue
                second_first = self.compose(
                    second,
                    first,
                )
                for third in self.morphisms():
                    if second.target() != third.source():
                        continue
                    third_second = self.compose(
                        third,
                        second,
                    )
                    left_association = self.compose(
                        third,
                        second_first,
                    )
                    right_association = self.compose(
                        third_second,
                        first,
                    )
                    if left_association is not right_association:
                        raise ValueError(
                            'the supplied composition table is not associative'
                        )

    def objects(self):
        return self._objects

    def morphisms(self):
        return tuple(
            self._morphisms_by_name.values()
        )

    def nonidentity_morphisms(self):
        return tuple(
            morphism
            for morphism in self.morphisms()
            if not morphism.is_identity()
        )

    def identity(self, obj):
        return self._identities[obj]

    def morphism(self, name):
        return self._morphisms_by_name[name]

    def hom(self, source, target):
        return tuple(
            morphism
            for morphism in self.morphisms()
            if morphism.source() == source
            and morphism.target() == target
        )

    def compose(self, left, right):
        if left.category() is not self:
            raise TypeError(
                'the left morphism is not in this finite category'
            )
        if right.category() is not self:
            raise TypeError(
                'the right morphism is not in this finite category'
            )
        if right.target() != left.source():
            raise TypeError(
                'the finite-category morphisms are not composable'
            )
        return self._composition_table[
            (left, right)
        ]

    def is_discrete(self):
        return not self.nonidentity_morphisms()

    def _repr_(self):
        return self._name


walking_span_regression_v2 = (
    FiniteSmallCategory.walking_span()
)
assert len(
    walking_span_regression_v2.objects()
) == 3
assert len(
    walking_span_regression_v2.nonidentity_morphisms()
) == 2
print('Installed finite indexing categories.')
print('walking span morphisms =', walking_span_regression_v2.morphisms())

Installed finite indexing categories.
walking span morphisms = (id_center, id_left, id_right, to_left, to_right)


## Finite diagram categories and natural transformations

For a finite small category $I$ and a Sage category $\mathcal C$, we construct the diagram category $[I,\mathcal C]$. Objects are functors $I\to\mathcal C$, and morphisms are natural transformations. Functoriality and naturality are checked using the proof-aware morphism equality of the target category.

In [11]:
def _categorical_morphisms_equal(
    category,
    left,
    right,
):
    if isinstance(left, ArrowMorphism):
        if not isinstance(right, ArrowMorphism):
            return False
        if left.domain().construction_category() is not (
            right.domain().construction_category()
        ):
            return False
        return left.domain().arrow_category().arrow_morphisms_equal(
            left,
            right,
        )
    return category.ArrowCategory().morphisms_equal(
        left,
        right,
    )


class DiagramCategoryFunctor(CovariantFunctorialConstruction):
    _functor_name = 'diagram'
    _functor_category = 'DiagramCategory'

    def __init__(self, index_category):
        self._index_category = index_category

    def index_category(self):
        return self._index_category


class DiagramCategoryConstruction(CovariantConstructionCategory):
    _functor_category = 'DiagramCategory'
    _base_category_class = (Category,)

    def __init__(self, category, index_category):
        if not isinstance(
            index_category,
            FiniteSmallCategory,
        ):
            raise TypeError(
                'the current diagram implementation requires a finite small category'
            )
        self._base_category = category
        self._args = (index_category,)
        self._index_category = index_category
        Category.__init__(self)

    def index_category(self):
        return self._index_category

    def _repr_object_names(self):
        return (
            f'{self._index_category}-shaped diagrams in '
            f'{self.base_category()._repr_object_names()}'
        )

    def diagram(self, object_map, morphism_map=None):
        return FiniteDiagram(
            self,
            object_map,
            morphism_map=morphism_map,
        )

    def constant_diagram(self, obj):
        return self.diagram({
            index_object: obj
            for index_object in self._index_category.objects()
        })


@cached_method
def _diagram_category(self, index_category):
    return DiagramCategoryConstruction(
        self,
        index_category,
    )


Category.DiagramCategory = _diagram_category


class FiniteDiagram(SageObject):
    def __init__(
        self,
        category,
        object_map,
        morphism_map=None,
    ):
        self._category = category
        self._index_category = category.index_category()
        self._object_map = dict(object_map)
        if set(self._object_map) != set(
            self._index_category.objects()
        ):
            raise ValueError(
                'the object map must be defined on every indexing object'
            )
        for value in self._object_map.values():
            if value not in category.base_category():
                raise TypeError(
                    'a diagram value is not an object of the target category'
                )

        supplied_morphism_map = morphism_map or {}
        self._morphism_map = {}
        for index_morphism in self._index_category.morphisms():
            source_value = self._object_map[
                index_morphism.source()
            ]
            target_value = self._object_map[
                index_morphism.target()
            ]
            if index_morphism.is_identity():
                image = Hom(
                    source_value,
                    source_value,
                ).identity()
            elif index_morphism in supplied_morphism_map:
                image = supplied_morphism_map[
                    index_morphism
                ]
            elif index_morphism.name() in supplied_morphism_map:
                image = supplied_morphism_map[
                    index_morphism.name()
                ]
            else:
                raise ValueError(
                    f'the image of {index_morphism} was not supplied'
                )
            if image.domain() is not source_value:
                raise ValueError(
                    f'the image of {index_morphism} has the wrong domain'
                )
            if image.codomain() is not target_value:
                raise ValueError(
                    f'the image of {index_morphism} has the wrong codomain'
                )
            self._morphism_map[
                index_morphism
            ] = image

        self._verify_functoriality()

    def category(self):
        return self._category

    def index_category(self):
        return self._index_category

    def target_category(self):
        return self._category.base_category()

    def object(self, index_object):
        return self._object_map[index_object]

    def morphism(self, index_morphism):
        return self._morphism_map[index_morphism]

    def objects(self):
        return dict(self._object_map)

    def morphisms(self):
        return dict(self._morphism_map)

    def _verify_functoriality(self):
        target_category = self.target_category()
        for right in self._index_category.morphisms():
            for left in self._index_category.morphisms():
                if right.target() != left.source():
                    continue
                composite = self._index_category.compose(
                    left,
                    right,
                )
                image_composite = self.morphism(
                    composite
                )
                composite_images = (
                    self.morphism(left)
                    * self.morphism(right)
                )
                status = _categorical_morphisms_equal(
                    target_category,
                    image_composite,
                    composite_images,
                )
                if status is False:
                    raise ValueError(
                        'the supplied data do not define a functor'
                    )
                if status is None:
                    raise NotImplementedError(
                        'functoriality could not be certified'
                    )

    def _Hom_(self, codomain, category=None):
        return NaturalTransformationHomset(
            self,
            codomain,
        )

    def limit(self, backend=None):
        return _FINITE_LIMIT_BACKENDS.dispatch(
            self,
            backend=backend,
        )

    def colimit(self, backend=None):
        return _FINITE_COLIMIT_BACKENDS.dispatch(
            self,
            backend=backend,
        )

    def _repr_(self):
        return (
            f'{self._index_category}-shaped diagram in '
            f'{self.target_category()}'
        )


class NaturalTransformationHomset(Homset):
    def __init__(self, domain, codomain):
        if domain.category() is not codomain.category():
            raise TypeError(
                'natural transformations require a common diagram category'
            )
        Homset.__init__(
            self,
            domain,
            codomain,
            category=domain.category(),
        )

    def _element_constructor_(self, components):
        return NaturalTransformation(
            self,
            components,
        )

    def identity(self):
        if self.domain() is not self.codomain():
            raise TypeError(
                'the identity transformation is defined only on an endomorphism homset'
            )
        return self({
            index_object: Hom(
                self.domain().object(index_object),
                self.domain().object(index_object),
            ).identity()
            for index_object in (
                self.domain().index_category().objects()
            )
        })


class NaturalTransformation(Element):
    def __init__(self, parent, components):
        self._components = dict(components)
        source_diagram = parent.domain()
        target_diagram = parent.codomain()
        index_category = source_diagram.index_category()
        if set(self._components) != set(
            index_category.objects()
        ):
            raise ValueError(
                'a natural transformation needs one component at every object'
            )
        for index_object, component in self._components.items():
            if component.domain() is not source_diagram.object(
                index_object
            ):
                raise ValueError(
                    'a natural-transformation component has the wrong domain'
                )
            if component.codomain() is not target_diagram.object(
                index_object
            ):
                raise ValueError(
                    'a natural-transformation component has the wrong codomain'
                )

        target_category = source_diagram.target_category()
        for index_morphism in index_category.nonidentity_morphisms():
            source_index = index_morphism.source()
            target_index = index_morphism.target()
            left_composite = (
                target_diagram.morphism(
                    index_morphism
                )
                * self._components[source_index]
            )
            right_composite = (
                self._components[target_index]
                * source_diagram.morphism(
                    index_morphism
                )
            )
            status = _categorical_morphisms_equal(
                target_category,
                left_composite,
                right_composite,
            )
            if status is False:
                raise ValueError(
                    'the supplied components are not natural'
                )
            if status is None:
                raise NotImplementedError(
                    'naturality could not be certified'
                )
        Element.__init__(self, parent)

    def domain(self):
        return self.parent().domain()

    def codomain(self):
        return self.parent().codomain()

    def component(self, index_object):
        return self._components[index_object]

    def components(self):
        return dict(self._components)

    def __mul__(self, right):
        if not isinstance(
            right,
            NaturalTransformation,
        ):
            return NotImplemented
        if right.codomain() is not self.domain():
            raise TypeError(
                'the natural transformations are not composable'
            )
        return Hom(
            right.domain(),
            self.codomain(),
        )({
            index_object: (
                self.component(index_object)
                * right.component(index_object)
            )
            for index_object in (
                self.domain().index_category().objects()
            )
        })

    def _repr_(self):
        return (
            f'Natural transformation from {self.domain()} '
            f'to {self.codomain()}'
        )


walking_arrow_diagram_shape_v2 = (
    FiniteSmallCategory.walking_arrow()
)
walking_arrow_diagram_category_v2 = (
    CommutativeRings().DiagramCategory(
        walking_arrow_diagram_shape_v2
    )
)
walking_arrow_diagram_v2 = (
    walking_arrow_diagram_category_v2.diagram(
        {
            'source': R_arrow_regression,
            'target': A_arrow_regression,
        },
        morphism_map={
            'arrow': eta_linear_arrow_regression,
        },
    )
)
identity_natural_transformation_v2 = Hom(
    walking_arrow_diagram_v2,
    walking_arrow_diagram_v2,
).identity()
assert identity_natural_transformation_v2.component(
    'source'
).is_identity()
assert identity_natural_transformation_v2.component(
    'target'
).is_identity()
print('Installed finite diagram categories and natural transformations.')
print('walking-arrow diagram =', walking_arrow_diagram_v2)

Installed finite diagram categories and natural transformations.
walking-arrow diagram = walking arrow-shaped diagram in Category of commutative rings


## Cones, cocones, limits, and colimits

For a finite diagram $D:I\to\mathcal C$, we construct cones and cocones as natural transformations from or to constant diagrams. Limit and colimit objects retain their universal mediator constructors, backend names, and certificates. Initial routed backends compute binary coproducts in ring coslices and the corresponding binary affine pullbacks in scheme slices.

In [12]:
class DiagramCone(SageObject):
    def __init__(self, diagram, apex, legs):
        self._diagram = diagram
        self._apex = apex
        self._constant_diagram = (
            diagram.category().constant_diagram(
                apex
            )
        )
        self._transformation = Hom(
            self._constant_diagram,
            diagram,
        )(legs)

    def diagram(self):
        return self._diagram

    def apex(self):
        return self._apex

    def leg(self, index_object):
        return self._transformation.component(
            index_object
        )

    def transformation(self):
        return self._transformation


class DiagramCocone(SageObject):
    def __init__(self, diagram, apex, legs):
        self._diagram = diagram
        self._apex = apex
        self._constant_diagram = (
            diagram.category().constant_diagram(
                apex
            )
        )
        self._transformation = Hom(
            diagram,
            self._constant_diagram,
        )(legs)

    def diagram(self):
        return self._diagram

    def apex(self):
        return self._apex

    def leg(self, index_object):
        return self._transformation.component(
            index_object
        )

    def transformation(self):
        return self._transformation


class LimitResult(SageObject):
    def __init__(
        self,
        diagram,
        cone,
        universal_builder,
        backend,
        certificate,
    ):
        if cone.diagram() is not diagram:
            raise ValueError(
                'the limit cone has the wrong diagram'
            )
        self._diagram = diagram
        self._cone = cone
        self._universal_builder = universal_builder
        self._backend = backend
        self._certificate = certificate

    def diagram(self):
        return self._diagram

    def cone(self):
        return self._cone

    def apex(self):
        return self._cone.apex()

    def backend(self):
        return self._backend

    def certificate(self):
        return self._certificate

    def universal_morphism(self, other_cone):
        if other_cone.diagram() is not self._diagram:
            raise ValueError(
                'the competing cone has the wrong diagram'
            )
        mediator = self._universal_builder(
            other_cone
        )
        target_category = (
            self._diagram.target_category()
        )
        for index_object in (
            self._diagram.index_category().objects()
        ):
            left = (
                self._cone.leg(index_object)
                * mediator
            )
            right = other_cone.leg(
                index_object
            )
            status = _categorical_morphisms_equal(
                target_category,
                left,
                right,
            )
            if status is False:
                raise ArithmeticError(
                    'the proposed limit mediator does not factor the cone'
                )
            if status is None:
                raise NotImplementedError(
                    'the limit mediator equations could not be certified'
                )
        return mediator


class ColimitResult(SageObject):
    def __init__(
        self,
        diagram,
        cocone,
        universal_builder,
        backend,
        certificate,
    ):
        if cocone.diagram() is not diagram:
            raise ValueError(
                'the colimit cocone has the wrong diagram'
            )
        self._diagram = diagram
        self._cocone = cocone
        self._universal_builder = universal_builder
        self._backend = backend
        self._certificate = certificate

    def diagram(self):
        return self._diagram

    def cocone(self):
        return self._cocone

    def apex(self):
        return self._cocone.apex()

    def backend(self):
        return self._backend

    def certificate(self):
        return self._certificate

    def universal_morphism(self, other_cocone):
        if other_cocone.diagram() is not self._diagram:
            raise ValueError(
                'the competing cocone has the wrong diagram'
            )
        mediator = self._universal_builder(
            other_cocone
        )
        target_category = (
            self._diagram.target_category()
        )
        for index_object in (
            self._diagram.index_category().objects()
        ):
            left = (
                mediator
                * self._cocone.leg(index_object)
            )
            right = other_cocone.leg(
                index_object
            )
            status = _categorical_morphisms_equal(
                target_category,
                left,
                right,
            )
            if status is False:
                raise ArithmeticError(
                    'the proposed colimit mediator does not factor the cocone'
                )
            if status is None:
                raise NotImplementedError(
                    'the colimit mediator equations could not be certified'
                )
        return mediator


_FINITE_LIMIT_BACKENDS = ConstructionBackendRegistry(
    'finite-diagram limit'
)
_FINITE_COLIMIT_BACKENDS = ConstructionBackendRegistry(
    'finite-diagram colimit'
)


def _binary_discrete_coslice_colimit_applies(
    diagram,
):
    index_category = diagram.index_category()
    if not index_category.is_discrete():
        return False
    if len(index_category.objects()) != 2:
        return False
    values = tuple(
        diagram.object(index_object)
        for index_object in index_category.objects()
    )
    return (
        all(
            isinstance(value, ArrowObject)
            and value.arrow_kind() == 'coslice'
            for value in values
        )
        and values[0].construction_category()
        is values[1].construction_category()
    )


def _binary_discrete_coslice_colimit(
    diagram,
):
    left_index, right_index = (
        diagram.index_category().objects()
    )
    left = diagram.object(left_index)
    right = diagram.object(right_index)
    coslice_category = (
        left.construction_category()
    )
    pushout = coslice_category.coproduct(
        left,
        right,
    )
    cocone = DiagramCocone(
        diagram,
        pushout.apex(),
        {
            left_index: (
                pushout.left_injection()
            ),
            right_index: (
                pushout.right_injection()
            ),
        },
    )

    def universal_builder(other_cocone):
        return pushout.universal_morphism(
            other_cocone.apex(),
            other_cocone.leg(left_index),
            other_cocone.leg(right_index),
        )

    return ColimitResult(
        diagram,
        cocone,
        universal_builder,
        backend=(
            'binary_discrete_coslice_coproduct'
        ),
        certificate=pushout.certificate(),
    )


_FINITE_COLIMIT_BACKENDS.register(
    'binary_discrete_coslice_coproduct',
    _binary_discrete_coslice_colimit_applies,
    _binary_discrete_coslice_colimit,
    priority=100,
)


def _binary_pullback_of_affine_schemes_applies(
    diagram,
):
    index_category = diagram.index_category()
    if not index_category.is_discrete():
        return False
    if len(index_category.objects()) != 2:
        return False
    values = tuple(
        diagram.object(index_object)
        for index_object in index_category.objects()
    )
    if not all(
        isinstance(value, ArrowObject)
        and value.arrow_kind() == 'slice'
        and hasattr(
            value,
            '_source_coslice_object',
        )
        for value in values
    ):
        return False
    return (
        values[0]
        ._source_coslice_object
        .construction_category()
        is values[1]
        ._source_coslice_object
        .construction_category()
    )


def _binary_pullback_of_affine_schemes(
    diagram,
):
    left_index, right_index = (
        diagram.index_category().objects()
    )
    left = diagram.object(left_index)
    right = diagram.object(right_index)
    ring_coslice_category = (
        left._source_coslice_object
        .construction_category()
    )
    ring_pushout = (
        ring_coslice_category.coproduct(
            left._source_coslice_object,
            right._source_coslice_object,
        )
    )
    pullback = Spec(
        ring_pushout
    )
    cone = DiagramCone(
        diagram,
        pullback.apex(),
        {
            left_index: (
                pullback.projection_to_left()
            ),
            right_index: (
                pullback.projection_to_right()
            ),
        },
    )

    def universal_builder(other_cone):
        return pullback.universal_morphism(
            other_cone.apex(),
            other_cone.leg(left_index),
            other_cone.leg(right_index),
        )

    return LimitResult(
        diagram,
        cone,
        universal_builder,
        backend=(
            'binary_pullback_of_affine_schemes'
        ),
        certificate=pullback.certificate(),
    )


_FINITE_LIMIT_BACKENDS.register(
    'binary_pullback_of_affine_schemes',
    _binary_pullback_of_affine_schemes_applies,
    _binary_pullback_of_affine_schemes,
    priority=100,
)

print('Installed cones, cocones, finite limits, and finite colimits.')

Installed cones, cocones, finite limits, and finite colimits.


### Finite-diagram K3 chart regression

We place the universal affine K3-cover chart and the exact free open in a binary discrete diagram in the parameter-ring coslice. We compute its colimit, apply `Spec`, place the resulting relative affine schemes in a binary discrete diagram, and compute its limit. We verify the identity mediators for both universal properties and evaluate the pulled-back branch function at the fixed origin.

In [13]:
binary_diagram_shape_v2 = FiniteSmallCategory.discrete(
    ('universal_chart', 'free_open'),
    name='binary discrete category',
)

ring_diagram_category_v2 = (
    ParameterCoslice_v2.DiagramCategory(
        binary_diagram_shape_v2
    )
)
ring_diagram_v2 = ring_diagram_category_v2.diagram(
    {
        'universal_chart': universal_chart_object_v2,
        'free_open': free_open_object_v2,
    }
)
ring_colimit_v2 = ring_diagram_v2.colimit()
ring_identity_mediator_v2 = (
    ring_colimit_v2.universal_morphism(
        ring_colimit_v2.cocone()
    )
)
ring_identity_v2 = Hom(
    ring_colimit_v2.apex(),
    ring_colimit_v2.apex(),
).identity()
ring_identity_status_v2 = (
    ring_colimit_v2.apex()
    .arrow_category()
    .arrow_morphisms_equal(
        ring_identity_mediator_v2,
        ring_identity_v2,
    )
)
assert ring_identity_status_v2 is True

universal_chart_scheme_v2 = Spec(
    universal_chart_object_v2
)
free_open_scheme_v2 = Spec(
    free_open_object_v2
)
scheme_slice_v2 = (
    universal_chart_scheme_v2
    .construction_category()
)
assert free_open_scheme_v2.construction_category() is (
    scheme_slice_v2
)
scheme_diagram_category_v2 = (
    scheme_slice_v2.DiagramCategory(
        binary_diagram_shape_v2
    )
)
scheme_diagram_v2 = scheme_diagram_category_v2.diagram(
    {
        'universal_chart': universal_chart_scheme_v2,
        'free_open': free_open_scheme_v2,
    }
)
scheme_limit_v2 = scheme_diagram_v2.limit()
scheme_identity_mediator_v2 = (
    scheme_limit_v2.universal_morphism(
        scheme_limit_v2.cone()
    )
)
scheme_identity_v2 = Hom(
    scheme_limit_v2.apex(),
    scheme_limit_v2.apex(),
).identity()
scheme_identity_status_v2 = (
    scheme_limit_v2.apex()
    .arrow_category()
    .arrow_morphisms_equal(
        scheme_identity_mediator_v2,
        scheme_identity_v2,
    )
)
assert scheme_identity_status_v2 is True

limit_ring_object_v2 = (
    scheme_limit_v2.apex()
    ._source_coslice_object
)
limit_chart_presentation_v2 = (
    limit_ring_object_v2.presentation(
        'monic_polynomial_algebra'
    )
)
limit_affine_ring_v2 = (
    limit_chart_presentation_v2.affine_ring()
)
limit_x0_v2, limit_x1_v2 = (
    limit_affine_ring_v2.gens()
)
limit_origin_value_v2 = (
    limit_chart_presentation_v2
    .branch_polynomial()
    .subs(
        {
            limit_x0_v2: 0,
            limit_x1_v2: 0,
        }
    )
)
assert limit_origin_value_v2.is_unit()

print(
    'ring colimit backend =',
    ring_colimit_v2.backend(),
)
print(
    'scheme limit backend =',
    scheme_limit_v2.backend(),
)
print(
    'ring universal identity mediator =',
    ring_identity_status_v2,
)
print(
    'scheme universal identity mediator =',
    scheme_identity_status_v2,
)
print(
    'fixed-origin branch value after the limit =',
    limit_origin_value_v2,
)
print(
    'fixed-origin branch value is a unit =',
    limit_origin_value_v2.is_unit(),
)

ring colimit backend = binary_discrete_coslice_coproduct
scheme limit backend = binary_pullback_of_affine_schemes
ring universal identity mediator = True
scheme universal identity mediator = True
fixed-origin branch value after the limit = a00
fixed-origin branch value is a unit = True


## Finite products of rings and principal-localization overlaps

We construct finite products in $\mathbf{CRing}$ using genuine ring projections and their universal product morphism. We also construct morphisms out of principal localizations by their universal property and register

$$
A_f\otimes_A A_g\cong A_{fg}
$$

as a coproduct backend in the ring coslice.

In [14]:
from sage.rings.morphism import RingHomomorphism


class ProductRingMorphism(RingHomomorphism):
    def __init__(
        self,
        component_morphisms,
        product_ring=None,
    ):
        component_morphisms = tuple(
            component_morphisms
        )
        if not component_morphisms:
            raise ValueError(
                'a nonempty finite product is required'
            )
        domain = component_morphisms[0].domain()
        if any(
            morphism.domain() is not domain
            for morphism in component_morphisms
        ):
            raise ValueError(
                'all component morphisms must have the same domain'
            )
        if product_ring is None:
            product_ring = cartesian_product(
                tuple(
                    morphism.codomain()
                    for morphism in component_morphisms
                )
            )
        factors = tuple(
            product_ring.cartesian_factors()
        )
        codomains = tuple(
            morphism.codomain()
            for morphism in component_morphisms
        )
        if len(factors) != len(codomains):
            raise ValueError(
                'the product ring has the wrong number of factors'
            )
        if any(
            factor is not codomain
            for factor, codomain in zip(
                factors,
                codomains,
            )
        ):
            raise ValueError(
                'the product factors do not match the component codomains'
            )
        self._component_morphisms = (
            component_morphisms
        )
        self._product_ring = product_ring
        RingHomomorphism.__init__(
            self,
            Hom(domain, product_ring),
        )

    def components(self):
        return self._component_morphisms

    def product_ring(self):
        return self._product_ring

    def _call_(self, element):
        return self._product_ring(
            tuple(
                morphism(element)
                for morphism in self._component_morphisms
            )
        )

    def _repr_defn(self):
        return 'Product of component ring morphisms'

    def is_identity(self):
        if self.domain() is not self.codomain():
            return False
        if self.domain() is not self._product_ring:
            return False
        if len(self._component_morphisms) != len(
            self._product_ring.cartesian_factors()
        ):
            return False
        for index, component in enumerate(
            self._component_morphisms
        ):
            if not isinstance(
                component,
                ProductProjectionRingMorphism,
            ):
                return False
            if component.domain() is not self._product_ring:
                return False
            if component.index() != index:
                return False
        return True

    def _composition_(self, right, homset):
        if right.codomain() is not self.domain():
            raise TypeError(
                'the product ring morphisms are not composable'
            )
        if _morphism_is_identity(right) is True:
            return self
        if self.is_identity():
            return right
        return ProductRingMorphism(
            tuple(
                component * right
                for component in self.components()
            ),
            product_ring=self.codomain(),
        )


class ProductProjectionRingMorphism(RingHomomorphism):
    def __init__(self, product_ring, index):
        factors = tuple(
            product_ring.cartesian_factors()
        )
        if index < 0 or index >= len(factors):
            raise IndexError(
                'the product projection index is out of range'
            )
        self._product_ring = product_ring
        self._index = index
        self._factor = factors[index]
        RingHomomorphism.__init__(
            self,
            Hom(product_ring, self._factor),
        )

    def index(self):
        return self._index

    def _call_(self, element):
        return element[self._index]

    def _repr_defn(self):
        return f'Projection to factor {self._index}'

    def _composition_(self, right, homset):
        if (
            isinstance(right, ProductRingMorphism)
            and right.codomain() is self.domain()
        ):
            return right.components()[self.index()]
        return RingHomomorphism._composition_(
            self,
            right,
            homset,
        )


class PrincipalLocalizationMorphism(RingHomomorphism):
    def __init__(
        self,
        localization_ring,
        target_ring,
        base_map,
        inverted_elements,
    ):
        inverted_elements = tuple(
            base_map.domain()(element)
            for element in inverted_elements
        )
        if base_map.domain() is not localization_ring.base_ring():
            raise ValueError(
                'the base map has the wrong domain for the localization'
            )
        if base_map.codomain() is not target_ring:
            raise ValueError(
                'the base map has the wrong codomain'
            )
        if not all(
            target_ring(base_map(element)).is_unit()
            for element in inverted_elements
        ):
            raise ValueError(
                'an inverted element does not map to a unit'
            )
        self._base_map = base_map
        self._inverted_elements = inverted_elements
        RingHomomorphism.__init__(
            self,
            Hom(localization_ring, target_ring),
        )

    def base_map(self):
        return self._base_map

    def inverted_elements(self):
        return self._inverted_elements

    def _call_(self, element):
        numerator_image = self._base_map(
            element.numerator()
        )
        denominator_image = self._base_map(
            element.denominator()
        )
        if not denominator_image.is_unit():
            raise ArithmeticError(
                'the denominator image is not a unit'
            )
        return (
            numerator_image
            * denominator_image.inverse_of_unit()
        )

    def _repr_defn(self):
        return (
            'Induced by the universal property of principal localization'
        )


def _localization_localization_backend_applies(
    category,
    left,
    right,
):
    left_presentation = left.presentation(
        'principal_localization'
    )
    right_presentation = right.presentation(
        'principal_localization'
    )
    return (
        left_presentation is not None
        and right_presentation is not None
        and left_presentation.base_ring()
        is category.base_object()
        and right_presentation.base_ring()
        is category.base_object()
    )


def _localization_localization_backend(
    category,
    left,
    right,
):
    left_presentation = left.presentation(
        'principal_localization'
    )
    right_presentation = right.presentation(
        'principal_localization'
    )
    base_ring = category.base_object()
    left_element = left_presentation.element()
    right_element = right_presentation.element()
    product_element = left_element * right_element
    product_presentation = (
        PrincipalLocalizationPresentation(
            base_ring,
            product_element,
        )
    )
    product_ring = product_presentation.ring()
    product_structure = (
        product_presentation.structure_morphism()
    )
    apex = category.coslice_object(
        product_structure
    ).register_presentation(
        product_presentation
    )

    arrow_category = category.arrow_category()
    base_certificate = (
        arrow_category
        ._morphism_equality_certificates.get(
            id(base_ring)
        )
    )
    if base_certificate is None:
        raise NotImplementedError(
            'the base-ring morphism equality has not been certified'
        )
    for presentation in (
        left_presentation,
        right_presentation,
        product_presentation,
    ):
        arrow_category.register_morphism_equality_certificate(
            presentation.equality_certificate(
                base_certificate
            )
        )

    left_injection = Hom(
        left,
        apex,
    )(
        PrincipalLocalizationMorphism(
            left_presentation.ring(),
            product_ring,
            product_structure,
            (left_element,),
        )
    )
    right_injection = Hom(
        right,
        apex,
    )(
        PrincipalLocalizationMorphism(
            right_presentation.ring(),
            product_ring,
            product_structure,
            (right_element,),
        )
    )

    def universal_builder(
        target,
        left_map,
        right_map,
    ):
        target_structure = target.arrow()
        if not target_structure(left_element).is_unit():
            raise ValueError(
                'the left localized element is not a unit in the target'
            )
        if not target_structure(right_element).is_unit():
            raise ValueError(
                'the right localized element is not a unit in the target'
            )
        return Hom(
            apex,
            target,
        )(
            PrincipalLocalizationMorphism(
                product_ring,
                target.underlying_object(),
                target_structure,
                (product_element,),
            )
        )

    return CoproductDiagram(
        category,
        left,
        right,
        apex,
        left_injection,
        right_injection,
        universal_builder,
        certificate=(
            'A_f tensor_A A_g identified with A_(fg) '
            'by the universal property of principal localization'
        ),
        backend=(
            'principal_localization_principal_localization'
        ),
    )


_COSLICE_COPRODUCT_BACKENDS.register(
    'principal_localization_principal_localization',
    _localization_localization_backend_applies,
    _localization_localization_backend,
    priority=110,
)


def _finite_discrete_ring_product_applies(diagram):
    return (
        diagram.index_category().is_discrete()
        and len(
            diagram.index_category().objects()
        ) >= 1
        and diagram.target_category().is_subcategory(
            CommutativeRings()
        )
    )


def _finite_discrete_ring_product(diagram):
    index_objects = (
        diagram.index_category().objects()
    )
    factors = tuple(
        diagram.object(index_object)
        for index_object in index_objects
    )
    product_ring = cartesian_product(factors)
    projections = tuple(
        ProductProjectionRingMorphism(
            product_ring,
            index,
        )
        for index in range(len(factors))
    )
    cone = DiagramCone(
        diagram,
        product_ring,
        {
            index_object: projections[index]
            for index, index_object in enumerate(
                index_objects
            )
        },
    )

    def universal_builder(other_cone):
        return ProductRingMorphism(
            tuple(
                other_cone.leg(index_object)
                for index_object in index_objects
            ),
            product_ring=product_ring,
        )

    return LimitResult(
        diagram,
        cone,
        universal_builder,
        backend='finite_product_of_commutative_rings',
        certificate=(
            'Cartesian product with canonical ring projections '
            'and componentwise universal morphism'
        ),
    )


_FINITE_LIMIT_BACKENDS.register(
    'finite_product_of_commutative_rings',
    _finite_discrete_ring_product_applies,
    _finite_discrete_ring_product,
    priority=90,
)

print('Installed finite products of rings and principal-localization overlaps.')

Installed finite products of rings and principal-localization overlaps.


## Routed composition, morphism equality, and affine coproducts

We normalize formal ring-map paths using product and localization universal properties. Morphisms into finite products are compared componentwise. Composition of morphisms of affine schemes is performed contravariantly on coordinate rings and then returned under `Spec`. Applying `Spec` to finite products of rings gives finite coproducts of affine schemes.

In [15]:
def _formal_ring_morphism_factors(morphism):
    if isinstance(morphism, FormalCompositeMap):
        return (
            _formal_ring_morphism_factors(
                morphism.first()
            )
            + _formal_ring_morphism_factors(
                morphism.then()
            )
        )
    return (morphism,)


def _try_simplify_ring_pair(
    later,
    earlier,
):
    if earlier.codomain() is not later.domain():
        raise TypeError(
            'the ring morphisms are not composable'
        )
    if _morphism_is_identity(earlier) is True:
        return later
    if _morphism_is_identity(later) is True:
        return earlier

    if (
        isinstance(later, PrincipalLocalizationMorphism)
        and isinstance(earlier, PrincipalLocalizationMorphism)
        and earlier.base_map().domain()
        is later.base_map().domain()
    ):
        return PrincipalLocalizationMorphism(
            earlier.domain(),
            later.codomain(),
            later.base_map(),
            earlier.inverted_elements(),
        )

    if (
        isinstance(
            later,
            ProductProjectionRingMorphism,
        )
        and isinstance(
            earlier,
            ProductRingMorphism,
        )
        and earlier.codomain() is later.domain()
    ):
        return earlier.components()[
            later.index()
        ]

    if isinstance(later, ProductRingMorphism):
        return ProductRingMorphism(
            tuple(
                _normalize_ring_morphism(
                    _compose_ring_morphisms_simplified(
                        component,
                        earlier,
                    )
                )
                for component in later.components()
            ),
            product_ring=later.codomain(),
        )

    candidate = later * earlier
    if candidate.domain() is candidate.codomain():
        identity = candidate.domain().Hom(
            candidate.domain()
        ).identity()
        if _morphism_is_identity(candidate) is True:
            return identity
        status = (
            CommutativeRings()
            .ArrowCategory()
            .morphisms_equal(
                candidate,
                identity,
            )
        )
        if status is True:
            return identity

    if not isinstance(candidate, FormalCompositeMap):
        return candidate
    return None


def _compose_ring_morphisms_simplified(
    later,
    earlier,
):
    simplified = _try_simplify_ring_pair(
        later,
        earlier,
    )
    if simplified is not None:
        return simplified
    return later * earlier


def _normalize_ring_morphism(morphism):
    factors = list(
        _formal_ring_morphism_factors(
            morphism
        )
    )
    normalized_factors = []
    for factor in factors:
        if isinstance(factor, ProductRingMorphism):
            factor = ProductRingMorphism(
                tuple(
                    _normalize_ring_morphism(
                        component
                    )
                    for component in factor.components()
                ),
                product_ring=factor.codomain(),
            )
            if factor.is_identity():
                factor = factor.domain().Hom(
                    factor.domain()
                ).identity()
        normalized_factors.append(factor)
    factors = normalized_factors

    changed = True
    while changed and len(factors) > 1:
        changed = False
        for index in range(len(factors) - 1):
            earlier = factors[index]
            later = factors[index + 1]
            simplified = _try_simplify_ring_pair(
                later,
                earlier,
            )
            if simplified is None:
                continue
            if isinstance(
                simplified,
                (FormalCompositeMap, ProductRingMorphism),
            ):
                simplified = _normalize_ring_morphism(
                    simplified
                )
            factors = (
                factors[:index]
                + [simplified]
                + factors[index + 2:]
            )
            changed = True
            break

    result = factors[0]
    for next_factor in factors[1:]:
        result = next_factor * result

    if isinstance(result, ProductRingMorphism):
        result = ProductRingMorphism(
            tuple(
                _normalize_ring_morphism(component)
                for component in result.components()
            ),
            product_ring=result.codomain(),
        )
        if result.is_identity():
            return result.domain().Hom(
                result.domain()
            ).identity()
    return result


def _coordinate_ring_morphism(morphism):
    if hasattr(morphism, 'ring_homomorphism'):
        ring_map = morphism.ring_homomorphism()
        if isinstance(
            ring_map,
            (FormalCompositeMap, ProductRingMorphism),
        ):
            return _normalize_ring_morphism(
                ring_map
            )
        return ring_map
    if isinstance(morphism, FormalCompositeMap):
        first_ring_map = _coordinate_ring_morphism(
            morphism.first()
        )
        remaining_ring_map = _coordinate_ring_morphism(
            morphism.then()
        )
        return _normalize_ring_morphism(
            first_ring_map * remaining_ring_map
        )
    if _morphism_is_identity(morphism) is True:
        coordinate_ring = (
            morphism.domain().coordinate_ring()
        )
        return coordinate_ring.Hom(
            coordinate_ring
        ).identity()
    raise NotImplementedError(
        'the morphism of affine schemes has no accessible coordinate-ring morphism'
    )


def _components_of_morphism_into_product(morphism):
    codomain = morphism.codomain()
    if not hasattr(codomain, 'cartesian_factors'):
        raise TypeError(
            'the codomain is not a finite Cartesian product'
        )
    normalized = _normalize_ring_morphism(
        morphism
    )
    if isinstance(normalized, ProductRingMorphism):
        return tuple(
            _normalize_ring_morphism(component)
            for component in normalized.components()
        )
    if _morphism_is_identity(normalized) is True:
        return tuple(
            ProductProjectionRingMorphism(
                codomain,
                index,
            )
            for index in range(
                len(codomain.cartesian_factors())
            )
        )
    return tuple(
        _normalize_ring_morphism(
            ProductProjectionRingMorphism(
                codomain,
                index,
            )
            * normalized
        )
        for index in range(
            len(codomain.cartesian_factors())
        )
    )


def _formal_ring_paths_equal(
    arrow_category,
    left,
    right,
):
    left_normal = _normalize_ring_morphism(
        left
    )
    right_normal = _normalize_ring_morphism(
        right
    )
    if _native_equality_proves_equal(
        left_normal,
        right_normal,
    ) is True:
        return True
    left_factors = _formal_ring_morphism_factors(
        left_normal
    )
    right_factors = _formal_ring_morphism_factors(
        right_normal
    )
    if len(left_factors) != len(right_factors):
        return None
    statuses = tuple(
        arrow_category.morphisms_equal(
            left_factor,
            right_factor,
        )
        for left_factor, right_factor in zip(
            left_factors,
            right_factors,
        )
    )
    if any(status is False for status in statuses):
        return False
    if all(status is True for status in statuses):
        return True
    return None


def _arrow_category_morphisms_equal_extended(
    self,
    left,
    right,
):
    if _native_equality_proves_equal(
        left,
        right,
    ) is True:
        return True
    if left.domain() is not right.domain():
        return False
    if left.codomain() is not right.codomain():
        return False
    if (
        _morphism_is_identity(left) is True
        and _morphism_is_identity(right) is True
    ):
        return True

    codomain = left.codomain()
    if (
        codomain in CommutativeRings()
        and hasattr(codomain, 'cartesian_factors')
    ):
        left_components = (
            _components_of_morphism_into_product(
                left
            )
        )
        right_components = (
            _components_of_morphism_into_product(
                right
            )
        )
        component_statuses = tuple(
            self.morphisms_equal(
                left_component,
                right_component,
            )
            for left_component, right_component in zip(
                left_components,
                right_components,
            )
        )
        if any(
            status is False
            for status in component_statuses
        ):
            return False
        if all(
            status is True
            for status in component_statuses
        ):
            return True
        return None

    certificate = self._morphism_equality_certificates.get(
        id(left.domain())
    )
    if certificate is not None:
        status = certificate.verify(
            left,
            right,
        )
        if status is not None:
            return status

    if (
        left.domain() in CommutativeRings()
        and left.codomain() in CommutativeRings()
        and (
            isinstance(left, FormalCompositeMap)
            or isinstance(right, FormalCompositeMap)
        )
    ):
        return _formal_ring_paths_equal(
            self,
            left,
            right,
        )
    return None


ArrowCategoryConstruction.morphisms_equal = (
    _arrow_category_morphisms_equal_extended
)


def _compose_categorical_morphisms(
    category,
    later,
    earlier,
):
    if earlier.codomain() is not later.domain():
        raise TypeError(
            'the categorical morphisms are not composable'
        )
    if _morphism_is_identity(earlier) is True:
        return later
    if _morphism_is_identity(later) is True:
        return earlier
    if isinstance(later, ArrowMorphism):
        return later * earlier
    if (
        hasattr(later.domain(), 'coordinate_ring')
        and hasattr(later.codomain(), 'coordinate_ring')
        and hasattr(earlier.domain(), 'coordinate_ring')
        and hasattr(earlier.codomain(), 'coordinate_ring')
    ):
        try:
            later_ring_map = _coordinate_ring_morphism(
                later
            )
            earlier_ring_map = _coordinate_ring_morphism(
                earlier
            )
        except NotImplementedError:
            pass
        else:
            return Spec(
                _normalize_ring_morphism(
                    earlier_ring_map
                    * later_ring_map
                )
            )
    return later * earlier


def _arrow_morphism_mul_routed(
    self,
    right,
):
    if not isinstance(right, ArrowMorphism):
        return NotImplemented
    if right.codomain() is not self.domain():
        raise TypeError(
            'the arrow morphisms are not composable'
        )
    try:
        if right.is_identity():
            return self
    except NotImplementedError:
        pass
    try:
        if self.is_identity():
            return right
    except NotImplementedError:
        pass
    homset = Hom(
        right.domain(),
        self.codomain(),
    )
    base_category = (
        self.domain()
        .construction_category()
        .base_category()
    )
    source_leg = _compose_categorical_morphisms(
        base_category,
        self.source_leg(),
        right.source_leg(),
    )
    target_leg = _compose_categorical_morphisms(
        base_category,
        self.target_leg(),
        right.target_leg(),
    )
    return ArrowMorphism(
        homset,
        source_leg,
        target_leg,
        square_verified=True,
    )


ArrowMorphism.__mul__ = _arrow_morphism_mul_routed


def _limit_universal_morphism_routed(
    self,
    other_cone,
):
    if other_cone.diagram() is not self._diagram:
        raise ValueError(
            'the competing cone has the wrong diagram'
        )
    mediator = self._universal_builder(
        other_cone
    )
    target_category = (
        self._diagram.target_category()
    )
    for index_object in (
        self._diagram.index_category().objects()
    ):
        left = _compose_categorical_morphisms(
            target_category,
            self._cone.leg(index_object),
            mediator,
        )
        right = other_cone.leg(
            index_object
        )
        status = _categorical_morphisms_equal(
            target_category,
            left,
            right,
        )
        if status is False:
            raise ArithmeticError(
                'the proposed limit mediator does not factor the cone'
            )
        if status is None:
            raise NotImplementedError(
                'the limit mediator equations could not be certified'
            )
    return mediator


def _colimit_universal_morphism_routed(
    self,
    other_cocone,
):
    if other_cocone.diagram() is not self._diagram:
        raise ValueError(
            'the competing cocone has the wrong diagram'
        )
    mediator = self._universal_builder(
        other_cocone
    )
    target_category = (
        self._diagram.target_category()
    )
    for index_object in (
        self._diagram.index_category().objects()
    ):
        left = _compose_categorical_morphisms(
            target_category,
            mediator,
            self._cocone.leg(index_object),
        )
        right = other_cocone.leg(
            index_object
        )
        status = _categorical_morphisms_equal(
            target_category,
            left,
            right,
        )
        if status is False:
            raise ArithmeticError(
                'the proposed colimit mediator does not factor the cocone'
            )
        if status is None:
            raise NotImplementedError(
                'the colimit mediator equations could not be certified'
            )
    return mediator


LimitResult.universal_morphism = (
    _limit_universal_morphism_routed
)
ColimitResult.universal_morphism = (
    _colimit_universal_morphism_routed
)


def _finite_discrete_affine_scheme_coproduct_applies(
    diagram,
):
    return (
        diagram.index_category().is_discrete()
        and len(
            diagram.index_category().objects()
        ) >= 1
        and diagram.target_category().is_subcategory(
            Schemes()
        )
        and all(
            hasattr(
                diagram.object(index_object),
                'coordinate_ring',
            )
            for index_object in (
                diagram.index_category().objects()
            )
        )
    )


def _finite_discrete_affine_scheme_coproduct(
    diagram,
):
    index_objects = (
        diagram.index_category().objects()
    )
    affine_schemes = tuple(
        diagram.object(index_object)
        for index_object in index_objects
    )
    coordinate_rings = tuple(
        scheme.coordinate_ring()
        for scheme in affine_schemes
    )
    ring_shape = FiniteSmallCategory.discrete(
        index_objects,
        name='coordinate-ring product shape',
    )
    ring_diagram = (
        CommutativeRings()
        .DiagramCategory(ring_shape)
        .diagram(
            {
                index_object: coordinate_ring
                for index_object, coordinate_ring in zip(
                    index_objects,
                    coordinate_rings,
                )
            }
        )
    )
    ring_product = ring_diagram.limit(
        backend='finite_product_of_commutative_rings'
    )
    coproduct_scheme = Spec(
        ring_product.apex()
    )
    scheme_arrow_category = Schemes().ArrowCategory()
    ring_arrow_category = CommutativeRings().ArrowCategory()
    for affine_scheme in (
        tuple(affine_schemes)
        + (coproduct_scheme,)
    ):
        scheme_arrow_category.register_morphism_equality_certificate(
            MorphismOfAffineSchemesEqualityCertificate(
                affine_scheme,
                ring_arrow_category,
            )
        )
    injections = {
        index_object: Spec(
            ring_product.cone().leg(index_object)
        )
        for index_object in index_objects
    }
    cocone = DiagramCocone(
        diagram,
        coproduct_scheme,
        injections,
    )

    def universal_builder(other_cocone):
        target_scheme = other_cocone.apex()
        if not hasattr(target_scheme, 'coordinate_ring'):
            raise NotImplementedError(
                'the current coproduct backend requires an affine target'
            )
        scheme_arrow_category.register_morphism_equality_certificate(
            MorphismOfAffineSchemesEqualityCertificate(
                target_scheme,
                ring_arrow_category,
            )
        )
        component_ring_maps = tuple(
            _coordinate_ring_morphism(
                other_cocone.leg(index_object)
            )
            for index_object in index_objects
        )
        product_ring_map = ProductRingMorphism(
            component_ring_maps,
            product_ring=ring_product.apex(),
        )
        return Spec(product_ring_map)

    return ColimitResult(
        diagram,
        cocone,
        universal_builder,
        backend='finite_coproduct_of_affine_schemes',
        certificate=(
            'Spec of the finite product of coordinate rings'
        ),
    )


_FINITE_COLIMIT_BACKENDS.register(
    'finite_coproduct_of_affine_schemes',
    _finite_discrete_affine_scheme_coproduct_applies,
    _finite_discrete_affine_scheme_coproduct,
    priority=95,
)

print('Installed routed composition, extensional equality, and affine coproducts.')

Installed routed composition, extensional equality, and affine coproducts.


## Principal affine covers and truncated Čech nerves

A finite principal affine cover is represented by the coproduct morphism

$$
\coprod_i \operatorname{Spec}(A_{f_i})\longrightarrow \operatorname{Spec}(A),
$$

with a Bézout certificate $\sum_i c_i f_i=1$. Its Čech nerve is constructed as a diagram from the opposite truncated simplex category. Degree $n$ is the coproduct of the intersections indexed by $(n+1)$-tuples, and every face and degeneracy map is induced by a principal-localization universal morphism.

In [16]:
from itertools import (
    combinations_with_replacement,
    product as cartesian_index_product,
)


def TruncatedSimplexCategoryOpposite(max_degree):
    max_degree = Integer(max_degree)
    if max_degree < 0:
        raise ValueError(
            'the truncation degree must be nonnegative'
        )
    objects = tuple(range(max_degree + 1))
    morphism_data = []
    name_by_data = {}

    for source in objects:
        for target in objects:
            for values in combinations_with_replacement(
                range(source + 1),
                target + 1,
            ):
                values = tuple(values)
                if (
                    source == target
                    and values == tuple(
                        range(source + 1)
                    )
                ):
                    name_by_data[
                        (source, target, values)
                    ] = ('identity', source)
                    continue
                name = (
                    'Delta_op',
                    source,
                    target,
                    values,
                )
                name_by_data[
                    (source, target, values)
                ] = name
                morphism_data.append(
                    (name, source, target)
                )

    compositions = {}
    for (
        right_name,
        right_source,
        right_target,
    ) in morphism_data:
        right_values = right_name[3]
        for (
            left_name,
            left_source,
            left_target,
        ) in morphism_data:
            if right_target != left_source:
                continue
            left_values = left_name[3]
            composite_values = tuple(
                right_values[value]
                for value in left_values
            )
            compositions[
                (left_name, right_name)
            ] = name_by_data[
                (
                    right_source,
                    left_target,
                    composite_values,
                )
            ]

    return FiniteSmallCategory(
        objects,
        morphisms=tuple(morphism_data),
        compositions=compositions,
        name=(
            'opposite of the simplex category '
            f'truncated in degrees 0 through {max_degree}'
        ),
    )


class PrincipalAffineOpenCoverCertificate(SageObject):
    def __init__(
        self,
        base_ring,
        indexed_elements,
        bezout_coefficients,
    ):
        self._base_ring = base_ring
        self._indexed_elements = {
            index: base_ring(element)
            for index, element in indexed_elements.items()
        }
        self._bezout_coefficients = {
            index: base_ring(
                bezout_coefficients[index]
            )
            for index in self._indexed_elements
        }
        self._bezout_sum = sum(
            self._bezout_coefficients[index]
            * self._indexed_elements[index]
            for index in self._indexed_elements
        )
        if self._bezout_sum != base_ring.one():
            raise ValueError(
                'the supplied coefficients do not certify a cover'
            )

    def base_ring(self):
        return self._base_ring

    def indices(self):
        return tuple(self._indexed_elements)

    def element(self, index):
        return self._indexed_elements[index]

    def coefficient(self, index):
        return self._bezout_coefficients[index]

    def bezout_sum(self):
        return self._bezout_sum

    def verifies_joint_surjectivity(self):
        return (
            self._bezout_sum
            == self._base_ring.one()
        )


class PrincipalAffineOpenCover(SageObject):
    def __init__(
        self,
        base_ring,
        indexed_elements,
        bezout_coefficients,
    ):
        self._certificate = (
            PrincipalAffineOpenCoverCertificate(
                base_ring,
                indexed_elements,
                bezout_coefficients,
            )
        )
        self._base_ring = base_ring
        self._target_scheme = Spec(base_ring)
        self._indices = self._certificate.indices()
        self._ring_coslice = (
            CommutativeRings().CosliceCategory(
                base_ring
            )
        )
        self._ring_coslice.register_morphism_equality_certificate(
            PolynomialGeneratorEqualityCertificate(
                base_ring
            )
        )
        self._presentations = {}
        self._ring_objects = {}
        self._scheme_objects = {}

        arrow_category = (
            self._ring_coslice.arrow_category()
        )
        base_certificate = (
            arrow_category
            ._morphism_equality_certificates[
                id(base_ring)
            ]
        )
        for index in self._indices:
            presentation = (
                PrincipalLocalizationPresentation(
                    base_ring,
                    self._certificate.element(index),
                )
            )
            arrow_category.register_morphism_equality_certificate(
                presentation.equality_certificate(
                    base_certificate
                )
            )
            ring_object = (
                self._ring_coslice
                .coslice_object(
                    presentation.structure_morphism()
                )
                .register_presentation(
                    presentation
                )
            )
            scheme_object = Spec(ring_object)
            self._presentations[index] = presentation
            self._ring_objects[index] = ring_object
            self._scheme_objects[index] = scheme_object

        self._scheme_slice = next(
            iter(self._scheme_objects.values())
        ).construction_category()
        self._index_category = (
            FiniteSmallCategory.discrete(
                self._indices,
                name=(
                    'index category of a finite '
                    'principal affine open cover'
                ),
            )
        )
        self._component_diagram = (
            Schemes()
            .DiagramCategory(
                self._index_category
            )
            .diagram(
                {
                    index: (
                        self._scheme_objects[index]
                        .underlying_object()
                    )
                    for index in self._indices
                }
            )
        )
        self._coproduct = (
            self._component_diagram.colimit(
                backend=(
                    'finite_coproduct_of_affine_schemes'
                )
            )
        )
        target_cocone = DiagramCocone(
            self._component_diagram,
            self._target_scheme,
            {
                index: (
                    self._scheme_objects[index]
                    .arrow()
                )
                for index in self._indices
            },
        )
        self._covering_morphism = (
            self._coproduct.universal_morphism(
                target_cocone
            )
        )
        self._object_over_target = (
            self._scheme_slice.slice_object(
                self._covering_morphism
            )
        )

    def certificate(self):
        return self._certificate

    def indices(self):
        return self._indices

    def target(self):
        return self._target_scheme

    def scheme_slice(self):
        return self._scheme_slice

    def ring_coslice(self):
        return self._ring_coslice

    def presentation(self, index):
        return self._presentations[index]

    def ring_object(self, index):
        return self._ring_objects[index]

    def scheme_object(self, index):
        return self._scheme_objects[index]

    def component_morphism(self, index):
        return self._scheme_objects[index].arrow()

    def component_diagram(self):
        return self._component_diagram

    def coproduct(self):
        return self._coproduct

    def covering_morphism(self):
        return self._covering_morphism

    def object_over_target(self):
        return self._object_over_target

    def is_open_cover(self):
        return (
            self._certificate
            .verifies_joint_surjectivity()
        )


class PrincipalCechLevel(SageObject):
    def __init__(
        self,
        cover,
        degree,
    ):
        self._cover = cover
        self._degree = Integer(degree)
        if self._degree < 0:
            raise ValueError(
                'the Čech degree must be nonnegative'
            )
        self._tuples = tuple(
            cartesian_index_product(
                cover.indices(),
                repeat=self._degree + 1,
            )
        )
        base_ring = (
            cover.certificate().base_ring()
        )
        self._presentations = {}
        self._ring_objects = {}
        self._scheme_objects = {}

        arrow_category = (
            cover.ring_coslice().arrow_category()
        )
        base_certificate = (
            arrow_category
            ._morphism_equality_certificates[
                id(base_ring)
            ]
        )
        for index_tuple in self._tuples:
            localization_element = prod(
                cover.certificate().element(index)
                for index in index_tuple
            )
            presentation = (
                PrincipalLocalizationPresentation(
                    base_ring,
                    localization_element,
                )
            )
            arrow_category.register_morphism_equality_certificate(
                presentation.equality_certificate(
                    base_certificate
                )
            )
            ring_object = (
                cover.ring_coslice()
                .coslice_object(
                    presentation.structure_morphism()
                )
                .register_presentation(
                    presentation
                )
            )
            scheme_object = Spec(ring_object)
            self._presentations[
                index_tuple
            ] = presentation
            self._ring_objects[
                index_tuple
            ] = ring_object
            self._scheme_objects[
                index_tuple
            ] = scheme_object

        self._component_index_category = (
            FiniteSmallCategory.discrete(
                self._tuples,
                name=(
                    'component index category of '
                    f'Čech level {self._degree}'
                ),
            )
        )
        self._component_diagram = (
            Schemes()
            .DiagramCategory(
                self._component_index_category
            )
            .diagram(
                {
                    index_tuple: (
                        self._scheme_objects[index_tuple]
                        .underlying_object()
                    )
                    for index_tuple in self._tuples
                }
            )
        )
        self._coproduct = (
            self._component_diagram.colimit(
                backend=(
                    'finite_coproduct_of_affine_schemes'
                )
            )
        )
        target_cocone = DiagramCocone(
            self._component_diagram,
            cover.target(),
            {
                index_tuple: (
                    self._scheme_objects[index_tuple]
                    .arrow()
                )
                for index_tuple in self._tuples
            },
        )
        self._structure_morphism = (
            self._coproduct.universal_morphism(
                target_cocone
            )
        )
        self._object_over_target = (
            cover.scheme_slice().slice_object(
                self._structure_morphism
            )
        )

    def cover(self):
        return self._cover

    def degree(self):
        return self._degree

    def tuples(self):
        return self._tuples

    def presentation(self, index_tuple):
        return self._presentations[index_tuple]

    def ring_object(self, index_tuple):
        return self._ring_objects[index_tuple]

    def scheme_object(self, index_tuple):
        return self._scheme_objects[index_tuple]

    def component_diagram(self):
        return self._component_diagram

    def coproduct(self):
        return self._coproduct

    def structure_morphism(self):
        return self._structure_morphism

    def object_over_target(self):
        return self._object_over_target


class TruncatedPrincipalCechNerve(SageObject):
    def __init__(
        self,
        cover,
        max_degree,
    ):
        self._cover = cover
        self._max_degree = Integer(max_degree)
        self._index_category = (
            TruncatedSimplexCategoryOpposite(
                self._max_degree
            )
        )
        self._levels = {
            degree: PrincipalCechLevel(
                cover,
                degree,
            )
            for degree in range(
                self._max_degree + 1
            )
        }
        self._morphism_cache = {}
        morphism_map = {
            index_morphism: (
                self._map_for_index_morphism(
                    index_morphism
                )
            )
            for index_morphism in (
                self._index_category
                .nonidentity_morphisms()
            )
        }
        self._diagram = (
            cover.scheme_slice()
            .DiagramCategory(
                self._index_category
            )
            .diagram(
                {
                    degree: (
                        self._levels[degree]
                        .object_over_target()
                    )
                    for degree in range(
                        self._max_degree + 1
                    )
                },
                morphism_map=morphism_map,
            )
        )

    def cover(self):
        return self._cover

    def max_degree(self):
        return self._max_degree

    def index_category(self):
        return self._index_category

    def level(self, degree):
        return self._levels[degree]

    def diagram(self):
        return self._diagram

    def _map_for_index_morphism(
        self,
        index_morphism,
    ):
        if index_morphism.is_identity():
            degree = index_morphism.source()
            return Hom(
                self._levels[degree]
                .object_over_target(),
                self._levels[degree]
                .object_over_target(),
            ).identity()
        if index_morphism in self._morphism_cache:
            return self._morphism_cache[
                index_morphism
            ]

        source_level = self._levels[
            index_morphism.source()
        ]
        target_level = self._levels[
            index_morphism.target()
        ]
        selection = index_morphism.name()[3]
        component_maps = {}

        for source_tuple in source_level.tuples():
            target_tuple = tuple(
                source_tuple[position]
                for position in selection
            )
            source_presentation = (
                source_level.presentation(
                    source_tuple
                )
            )
            target_presentation = (
                target_level.presentation(
                    target_tuple
                )
            )
            if (
                target_presentation.ring()
                is source_presentation.ring()
            ):
                ring_map = Hom(
                    source_presentation.ring(),
                    source_presentation.ring(),
                ).identity()
            else:
                ring_map = PrincipalLocalizationMorphism(
                    target_presentation.ring(),
                    source_presentation.ring(),
                    source_presentation.structure_morphism(),
                    (
                        target_presentation.element(),
                    ),
                )
            component_scheme_map = Spec(
                ring_map
            )
            target_injection = (
                target_level.coproduct()
                .cocone()
                .leg(target_tuple)
            )
            component_maps[source_tuple] = (
                _compose_categorical_morphisms(
                    Schemes(),
                    target_injection,
                    component_scheme_map,
                )
            )

        component_cocone = DiagramCocone(
            source_level.component_diagram(),
            target_level.coproduct().apex(),
            component_maps,
        )
        absolute_map = (
            source_level.coproduct()
            .universal_morphism(
                component_cocone
            )
        )
        slice_map = Hom(
            source_level.object_over_target(),
            target_level.object_over_target(),
        )(absolute_map)
        self._morphism_cache[
            index_morphism
        ] = slice_map
        return slice_map


def _verify_functoriality_routed(self):
    target_category = self.target_category()
    for right in self._index_category.morphisms():
        for left in self._index_category.morphisms():
            if right.target() != left.source():
                continue
            composite = self._index_category.compose(
                left,
                right,
            )
            image_composite = self.morphism(
                composite
            )
            composite_images = (
                _compose_categorical_morphisms(
                    target_category,
                    self.morphism(left),
                    self.morphism(right),
                )
            )
            status = _categorical_morphisms_equal(
                target_category,
                image_composite,
                composite_images,
            )
            if status is False:
                raise ValueError(
                    'the supplied data do not define a functor'
                )
            if status is None:
                raise NotImplementedError(
                    'functoriality could not be certified'
                )


FiniteDiagram._verify_functoriality = (
    _verify_functoriality_routed
)

print('Installed principal affine covers and truncated Čech nerves.')

Installed principal affine covers and truncated Čech nerves.


## Monic algebras under principal-open restriction

For a commutative ring $A$, an element $f\in A$, and a monic algebra

$$
B=A[z]/(z^n-s),
$$

we construct its base change

$$
B\otimes_A A_f\cong A_f[z]/(z^n-s|_{A_f}).
$$

This backend is used below to restrict the universal degree-two K3-cover algebra to the members and overlaps of a principal affine cover.

In [17]:
class MonicAlgebraEqualityCertificate(
    MorphismEqualityCertificate
):
    def __init__(
        self,
        presentation,
        base_certificate,
    ):
        self._presentation = presentation
        self._domain = presentation.algebra()
        self._base_certificate = base_certificate

    def domain(self):
        return self._domain

    def verify(self, left, right):
        if left.domain() is not self._domain:
            return False
        if right.domain() is not self._domain:
            return False
        if left.codomain() is not right.codomain():
            return False
        if left(self._domain.gen()) != right(
            self._domain.gen()
        ):
            return False
        left_on_base = (
            left
            * self._presentation.structure_morphism()
        )
        right_on_base = (
            right
            * self._presentation.structure_morphism()
        )
        if _native_equality_proves_equal(
            left_on_base,
            right_on_base,
        ) is True:
            return True
        return self._base_certificate.verify(
            left_on_base,
            right_on_base,
        )


class MonicAlgebraPresentation(SageObject):
    kind = 'monic_algebra'

    def __init__(
        self,
        base_ring,
        cover_variable_name,
        branch_element,
        degree=2,
        quotient_name=None,
    ):
        self._base_ring = base_ring
        self._cover_variable_name = str(
            cover_variable_name
        )
        self._branch_element = base_ring(
            branch_element
        )
        self._degree = Integer(degree)
        self._polynomial_ring = PolynomialRing(
            base_ring,
            names=(self._cover_variable_name,),
        )
        cover_coordinate = self._polynomial_ring.gen()
        self._modulus = (
            cover_coordinate**self._degree
            - self._polynomial_ring(
                self._branch_element
            )
        )
        if not self._modulus.is_monic():
            raise ValueError(
                'the defining polynomial must be monic'
            )
        if quotient_name is None:
            quotient_name = (
                f'{self._cover_variable_name}bar'
            )
        self._algebra = self._polynomial_ring.quotient(
            self._modulus,
            names=(quotient_name,),
        )
        self._structure_morphism = (
            self._algebra.coerce_map_from(
                base_ring
            )
        )
        if self._structure_morphism is None:
            raise ValueError(
                'the monic algebra has no base structure morphism'
            )

    def base_ring(self):
        return self._base_ring

    def branch_element(self):
        return self._branch_element

    def cover_variable_name(self):
        return self._cover_variable_name

    def degree(self):
        return self._degree

    def polynomial_ring(self):
        return self._polynomial_ring

    def modulus(self):
        return self._modulus

    def algebra(self):
        return self._algebra

    def structure_morphism(self):
        return self._structure_morphism

    def equality_certificate(self, base_certificate):
        return MonicAlgebraEqualityCertificate(
            self,
            base_certificate,
        )


def _monic_algebra_localization_backend_applies(
    category,
    left,
    right,
):
    left_monic = left.presentation(
        'monic_algebra'
    )
    right_monic = right.presentation(
        'monic_algebra'
    )
    left_local = left.presentation(
        'principal_localization'
    )
    right_local = right.presentation(
        'principal_localization'
    )
    return bool(
        (
            left_monic is not None
            and right_local is not None
        )
        or
        (
            right_monic is not None
            and left_local is not None
        )
    )


def _monic_algebra_localization_backend(
    category,
    left,
    right,
):
    swapped = False
    if left.presentation(
        'monic_algebra'
    ) is None:
        left, right = right, left
        swapped = True

    monic = left.presentation(
        'monic_algebra'
    )
    localization = right.presentation(
        'principal_localization'
    )
    base_ring = category.base_object()
    if monic.base_ring() is not base_ring:
        raise ValueError(
            'the monic algebra has the wrong base ring'
        )
    if localization.base_ring() is not base_ring:
        raise ValueError(
            'the localization has the wrong base ring'
        )

    arrow_category = category.arrow_category()
    base_certificate = (
        arrow_category
        ._morphism_equality_certificates[
            id(base_ring)
        ]
    )
    localized_base = localization.ring()
    localized_structure = (
        localization.structure_morphism()
    )
    localized_monic = MonicAlgebraPresentation(
        localized_base,
        monic.cover_variable_name(),
        localized_structure(
            monic.branch_element()
        ),
        degree=monic.degree(),
        quotient_name=(
            f'{monic.cover_variable_name()}bar_bc'
        ),
    )
    localized_algebra = (
        localized_monic.algebra()
    )
    localized_algebra_structure = (
        localized_monic.structure_morphism()
    )
    apex_structure = (
        localized_algebra_structure
        * localized_structure
    )
    apex = category.coslice_object(
        apex_structure
    ).register_presentation(
        localized_monic
    )

    arrow_category.register_morphism_equality_certificate(
        monic.equality_certificate(
            base_certificate
        )
    )
    localization_certificate = (
        localization.equality_certificate(
            base_certificate
        )
    )
    arrow_category.register_morphism_equality_certificate(
        localization_certificate
    )
    arrow_category.register_morphism_equality_certificate(
        localized_monic.equality_certificate(
            localization_certificate
        )
    )

    source_algebra = monic.algebra()
    source_injection_ring_map = (
        source_algebra.hom(
            (localized_algebra.gen(),),
            localized_algebra,
            base_map=apex_structure,
        )
    )
    localization_injection_ring_map = (
        localized_algebra_structure
    )
    left_injection = Hom(
        left,
        apex,
    )(source_injection_ring_map)
    right_injection = Hom(
        right,
        apex,
    )(localization_injection_ring_map)

    def universal_builder(
        target,
        left_map,
        right_map,
    ):
        target_ring = target.underlying_object()
        mediator_ring_map = (
            localized_algebra.hom(
                (
                    left_map.target_leg()(
                        source_algebra.gen()
                    ),
                ),
                target_ring,
                base_map=right_map.target_leg(),
            )
        )
        return Hom(
            apex,
            target,
        )(mediator_ring_map)

    diagram = CoproductDiagram(
        category,
        left,
        right,
        apex,
        left_injection,
        right_injection,
        universal_builder,
        certificate=(
            'base change of a monic algebra along a principal localization'
        ),
        backend=(
            'monic_algebra_principal_localization'
        ),
    )
    if not swapped:
        return diagram
    return CoproductDiagram(
        category,
        right,
        left,
        apex,
        right_injection,
        left_injection,
        lambda target, first_map, second_map: (
            universal_builder(
                target,
                second_map,
                first_map,
            )
        ),
        certificate=diagram.certificate(),
        backend=diagram.backend(),
    )


_COSLICE_COPRODUCT_BACKENDS.register(
    'monic_algebra_principal_localization',
    _monic_algebra_localization_backend_applies,
    _monic_algebra_localization_backend,
    priority=120,
)

print('Installed monic algebras under principal-open restriction.')

Installed monic algebras under principal-open restriction.


### Čech and K3 overlap regressions

First, for $\operatorname{Spec}\mathbf Q[t]$ covered by $D(t)$ and $D(1-t)$, we construct the $2$-truncated Čech nerve and verify every simplicial identity. Then, on the universal K3 affine base chart, we construct the two-open principal cover $D(x_0)\cup D(1-x_0)$, all four degree-one overlaps, and the restrictions of the monic degree-two cover algebra. We compare the branch element on the mixed overlap through both restriction paths.

In [18]:
A_cech_regression_v2 = PolynomialRing(
    QQ,
    names=('t_cech_regression_v2',),
)
t_cech_regression_v2 = (
    A_cech_regression_v2.gen()
)
principal_cover_regression_v2 = (
    PrincipalAffineOpenCover(
        A_cech_regression_v2,
        {
            'D_t': t_cech_regression_v2,
            'D_1_minus_t': (
                1 - t_cech_regression_v2
            ),
        },
        {
            'D_t': 1,
            'D_1_minus_t': 1,
        },
    )
)
cech_nerve_regression_v2 = (
    TruncatedPrincipalCechNerve(
        principal_cover_regression_v2,
        2,
    )
)
cech_level_counts_regression_v2 = tuple(
    len(
        cech_nerve_regression_v2
        .level(degree)
        .tuples()
    )
    for degree in range(3)
)
assert principal_cover_regression_v2.is_open_cover()
assert cech_level_counts_regression_v2 == (
    2,
    4,
    8,
)

K3_base_cover_v2 = PrincipalAffineOpenCover(
    AffineChartRing_v2,
    {
        'x0_nonzero': x0_v2,
        'one_minus_x0_nonzero': (
            1 - x0_v2
        ),
    },
    {
        'x0_nonzero': 1,
        'one_minus_x0_nonzero': 1,
    },
)
K3_cech_level0_v2 = PrincipalCechLevel(
    K3_base_cover_v2,
    0,
)
K3_cech_level1_v2 = PrincipalCechLevel(
    K3_base_cover_v2,
    1,
)
assert K3_base_cover_v2.is_open_cover()
assert len(K3_cech_level0_v2.tuples()) == 2
assert len(K3_cech_level1_v2.tuples()) == 4

K3_chart_coslice_v2 = (
    CommutativeRings().CosliceCategory(
        AffineChartRing_v2
    )
)
K3_chart_coslice_v2.register_morphism_equality_certificate(
    PolynomialGeneratorEqualityCertificate(
        AffineChartRing_v2
    )
)
K3_monic_chart_v2 = MonicAlgebraPresentation(
    AffineChartRing_v2,
    'z00_chart',
    branch_function_00_v2,
    degree=2,
    quotient_name='zbar00_chart',
)
K3_cover_object_v2 = (
    K3_chart_coslice_v2.coslice_object(
        K3_monic_chart_v2.structure_morphism()
    ).register_presentation(
        K3_monic_chart_v2
    )
)

K3_open0_presentation_v2 = (
    K3_base_cover_v2.presentation(
        'x0_nonzero'
    )
)
K3_open1_presentation_v2 = (
    K3_base_cover_v2.presentation(
        'one_minus_x0_nonzero'
    )
)
K3_overlap_tuple_v2 = (
    'x0_nonzero',
    'one_minus_x0_nonzero',
)
K3_overlap_presentation_v2 = (
    K3_cech_level1_v2.presentation(
        K3_overlap_tuple_v2
    )
)

K3_open0_object_v2 = (
    K3_chart_coslice_v2.coslice_object(
        K3_open0_presentation_v2
        .structure_morphism()
    ).register_presentation(
        K3_open0_presentation_v2
    )
)
K3_open1_object_v2 = (
    K3_chart_coslice_v2.coslice_object(
        K3_open1_presentation_v2
        .structure_morphism()
    ).register_presentation(
        K3_open1_presentation_v2
    )
)
K3_overlap_object_v2 = (
    K3_chart_coslice_v2.coslice_object(
        K3_overlap_presentation_v2
        .structure_morphism()
    ).register_presentation(
        K3_overlap_presentation_v2
    )
)

K3_cover_on_open0_v2 = (
    K3_chart_coslice_v2.coproduct(
        K3_cover_object_v2,
        K3_open0_object_v2,
    )
)
K3_cover_on_open1_v2 = (
    K3_chart_coslice_v2.coproduct(
        K3_cover_object_v2,
        K3_open1_object_v2,
    )
)
K3_cover_on_overlap_v2 = (
    K3_chart_coslice_v2.coproduct(
        K3_cover_object_v2,
        K3_overlap_object_v2,
    )
)

K3_monic_open0_v2 = (
    K3_cover_on_open0_v2
    .apex()
    .presentation('monic_algebra')
)
K3_monic_open1_v2 = (
    K3_cover_on_open1_v2
    .apex()
    .presentation('monic_algebra')
)
K3_monic_overlap_v2 = (
    K3_cover_on_overlap_v2
    .apex()
    .presentation('monic_algebra')
)

K3_overlap_ring_v2 = (
    K3_monic_overlap_v2.base_ring()
)
K3_overlap_structure_v2 = (
    K3_overlap_presentation_v2
    .structure_morphism()
)
K3_open0_to_overlap_v2 = (
    PrincipalLocalizationMorphism(
        K3_monic_open0_v2.base_ring(),
        K3_overlap_ring_v2,
        K3_overlap_structure_v2,
        (
            K3_base_cover_v2
            .certificate()
            .element('x0_nonzero'),
        ),
    )
)
K3_open1_to_overlap_v2 = (
    PrincipalLocalizationMorphism(
        K3_monic_open1_v2.base_ring(),
        K3_overlap_ring_v2,
        K3_overlap_structure_v2,
        (
            K3_base_cover_v2
            .certificate()
            .element(
                'one_minus_x0_nonzero'
            ),
        ),
    )
)
K3_branch_from_open0_v2 = (
    K3_open0_to_overlap_v2(
        K3_monic_open0_v2.branch_element()
    )
)
K3_branch_from_open1_v2 = (
    K3_open1_to_overlap_v2(
        K3_monic_open1_v2.branch_element()
    )
)
K3_branch_on_overlap_v2 = (
    K3_monic_overlap_v2.branch_element()
)

assert (
    K3_branch_from_open0_v2
    == K3_branch_on_overlap_v2
)
assert (
    K3_branch_from_open1_v2
    == K3_branch_on_overlap_v2
)
assert K3_monic_open0_v2.modulus().degree() == 2
assert K3_monic_open1_v2.modulus().degree() == 2
assert K3_monic_overlap_v2.modulus().degree() == 2

print(
    'small Čech level counts =',
    cech_level_counts_regression_v2,
)
print(
    'small 2-truncated simplicial identities verified =',
    True,
)
print(
    'K3 cover component counts in degrees 0 and 1 =',
    (
        len(K3_cech_level0_v2.tuples()),
        len(K3_cech_level1_v2.tuples()),
    ),
)
print(
    'K3 open backends =',
    (
        K3_cover_on_open0_v2.backend(),
        K3_cover_on_open1_v2.backend(),
        K3_cover_on_overlap_v2.backend(),
    ),
)
print(
    'K3 branch agrees from open 0 =',
    K3_branch_from_open0_v2
    == K3_branch_on_overlap_v2,
)
print(
    'K3 branch agrees from open 1 =',
    K3_branch_from_open1_v2
    == K3_branch_on_overlap_v2,
)
print(
    'K3 overlap cover degree =',
    K3_monic_overlap_v2.modulus().degree(),
)

small Čech level counts = (2, 4, 8)
small 2-truncated simplicial identities verified = True
K3 cover component counts in degrees 0 and 1 = (2, 4)
K3 open backends = ('monic_algebra_principal_localization', 'monic_algebra_principal_localization', 'monic_algebra_principal_localization')
K3 branch agrees from open 0 = True
K3 branch agrees from open 1 = True
K3 overlap cover degree = 2
